In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
import joblib
import warnings
import re
from datetime import datetime

warnings.filterwarnings('ignore')

# Custom transformers for specific data processing tasks
class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    """Extract features from date columns"""
    def __init__(self, date_columns):
        self.date_columns = date_columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.date_columns:
            if col in X.columns:
                X[col] = pd.to_datetime(X[col], errors='coerce')
                X[f'{col}_month'] = X[col].dt.month
                X[f'{col}_dayofweek'] = X[col].dt.dayofweek
                X[f'{col}_year'] = X[col].dt.year
                # Drop original date column
                X = X.drop(columns=[col])
        return X

class IncomeBandEncoder(BaseEstimator, TransformerMixin):
    """Encode income bands to numerical values"""
    def __init__(self):
        self.income_mapping = {
            '50,000 or Below': 0,
            '50,000 to 100,000': 1,
            '100,000 to 200,000': 2,
            '200,000 to 300,000': 3,
            '300,000 to 500,000': 4,
            '500,000 or Above': 5
        }

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if 'Income_Band_SGD' in X.columns:
            X['Income_Band_SGD'] = X['Income_Band_SGD'].map(self.income_mapping)
        return X

class BooleanConverter(BaseEstimator, TransformerMixin):
    """Convert various boolean representations to proper booleans"""
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        bool_columns = ['Partial_Payment_Indicator', 'Repayment_Irregularity_Flags',
                       'Mobile_Number_Active_Status', 'Email_Activity', 'Do_Not_Call_Registry_Data',
                       'WhatsApp_OTT_usage_Indicator', 'Overdraft_or_Low_Balance_Flag',
                       'Delinquency_on_other_Loans']

        for col in bool_columns:
            if col in X.columns:
                X[col] = X[col].astype(str).str.lower().replace({
                    'true': True, 'false': False, '1': True, '0': False,
                    'yes': True, 'no': False
                }).astype(bool)
        return X

class AddressProcessor(BaseEstimator, TransformerMixin):
    """Extract area and pincode from address column but don't encode them"""
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if 'Address' in X.columns:
            # Extract area and pincode from address (these will be kept as passthrough, not encoded)
            extracted_data = X['Address'].apply(self._extract_area_and_pincode)
            X['Area_From_Address'] = extracted_data.apply(lambda x: x[0])
            X['Pincode_From_Address'] = extracted_data.apply(lambda x: x[1])

        return X

    def _extract_area_and_pincode(self, address):
        """Extract area name and pincode from address"""
        if pd.isna(address):
            return (None, None)

        # Common Singapore areas (from your data generation code)
        singapore_areas = [
            "Raffles Place", "Marina Bay", "Suntec City", "Harbourfront",
            "Serangoon Garden", "Marine Parade", "Bukit Timah", "Orchard",
            "Tanjong Pagar", "Chinatown", "Little India", "Kampong Glam",
            "Bugis", "Dhoby Ghaut", "Somerset", "City Hall",
            "Lavender", "Kallang", "Geylang", "Eunos",
            "Bedok", "Tampines", "Pasir Ris", "Simei",
            "Jurong East", "Jurong West", "Clementi", "Bukit Batok",
            "Bukit Panjang", "Choa Chu Kang", "Woodlands", "Yishun",
            "Sembawang", "Ang Mo Kio", "Bishan", "Toa Payoh",
            "Serangoon", "Hougang", "Punggol", "Sengkang"
        ]

        # Look for area names in the address
        address_lower = address.lower()
        area_found = None

        for area in singapore_areas:
            if area.lower() in address_lower:
                area_found = area
                break

        # Extract pincode (6-digit Singapore pincode)
        pincode_match = re.search(r'(\d{6})', address)
        pincode = pincode_match.group(1) if pincode_match else None

        # If no area found using exact match, try to extract from address structure
        if area_found is None:
            # Try to extract the word before "Singapore" or look for common patterns
            match = re.search(r',\s*([^,]+?),\s*Singapore', address, re.IGNORECASE)
            if match:
                potential_area = match.group(1).strip()
                # Check if this potential area is in our list
                for area in singapore_areas:
                    if area.lower() in potential_area.lower():
                        area_found = area
                        break
                else:
                    area_found = potential_area  # Use as-is if not in list

        return (area_found, pincode)

class EmailValidator(BaseEstimator, TransformerMixin):
    """Validate email addresses and extract domain information"""
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if 'Email_ID' in X.columns:
            # Validate email format
            X['Email_Valid_Format'] = X['Email_ID'].apply(self._validate_email_format)

            # Extract domain
            X['Email_Domain'] = X['Email_ID'].apply(self._extract_domain)

            # Check if domain is legitimate (common email providers)
            X['Email_Domain_Legitimate'] = X['Email_Domain'].apply(self._is_legitimate_domain)

            # Check for disposable email domains
            X['Email_Disposable'] = X['Email_Domain'].apply(self._is_disposable_domain)

        return X

    def _validate_email_format(self, email):
        """Validate basic email format"""
        if pd.isna(email) or not isinstance(email, str):
            return False

        email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        return bool(re.match(email_pattern, email))

    def _extract_domain(self, email):
        """Extract domain from email address"""
        if pd.isna(email) or not isinstance(email, str) or '@' not in email:
            return None

        return email.split('@')[1].lower()

    def _is_legitimate_domain(self, domain):
        """Check if domain is from a legitimate email provider"""
        if pd.isna(domain):
            return False

        legitimate_domains = [
            'gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'live.com',
            'icloud.com', 'protonmail.com', 'zoho.com', 'aol.com', 'mail.com',
            'gmail.com.sg', 'yahoo.com.sg', 'hotmail.com.sg', 'singnet.com.sg',
            'starhub.net.sg', 'pacific.net.sg'
        ]

        return domain in legitimate_domains

    def _is_disposable_domain(self, domain):
        """Check if domain is from a disposable email service"""
        if pd.isna(domain):
            return False

        disposable_domains = [
            'tempmail.com', '10minutemail.com', 'guerrillamail.com',
            'mailinator.com', 'yopmail.com', 'trashmail.com',
            'disposableemail.com', 'fakeinbox.com', 'temp-mail.org'
        ]

        return domain in disposable_domains

class DataLoader(BaseEstimator, TransformerMixin):
    """Load and initial data processing"""
    def __init__(self, file_path, chunk_size=10000):
        self.file_path = file_path
        self.chunk_size = chunk_size

    def fit(self, X, y=None):
        return self

    def transform(self, X=None):
        """Load data from CSV file"""
        print("Loading data...")

        try:
            # Load data as regular CSV (no compression)
            df = pd.read_csv(self.file_path)
            print(f"Successfully loaded {len(df):,} rows")
            return df
        except Exception as e:
            print(f"Error loading data: {e}")
            # Return empty DataFrame with expected structure
            return pd.DataFrame()

class ColumnDropper(BaseEstimator, TransformerMixin):
    """Drop specified columns from the dataset"""
    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Drop columns if they exist
        columns_present = [col for col in self.columns_to_drop if col in X.columns]
        X = X.drop(columns=columns_present)
        return X

class SingaporeLoanDataPipeline:
    def __init__(self, file_path):
        self.file_path = file_path
        self.pipeline = None
        self.processed_data = None
        self.column_names = None
        self._build_pipeline()

    def _get_numeric_columns(self):
        """Define numeric columns for preprocessing"""
        return ['Age', 'Loan_Amount_SGD', 'Outstanding_Balance_SGD', 'Day_Past_Due',
                'Tenure', 'Interest_Rate', 'Current_EMI_SGD', 'Number_of_Past_Payments',
                'Amount_Paid_Each_Month_SGD', 'Missed_Payments_Count',
                'Contact_History_Call_Attempts', 'Contact_History_SMS', 'Contact_History_WhatsApp',
                'Contact_History_EmailLogs', 'No_of_Attempts', 'Average_Handling_Time',
                'Credit_Score', 'Recent_Inquiries', 'Loan_Exposure_Across_Banks',
                'Recent_Score_Change', 'Unemployeement_rate_region', 'Inflation_Rate',
                'Interest_Rate_Trend', 'Economic_Stress_Index', 'Income_Band_SGD',
                'Utility_Spend_SGD', 'Shopping_Spend_SGD', 'Entertainment_Spend_SGD',
                'Health_Spend_SGD', 'Education_Spend_SGD', 'Travel_Spend_SGD',
                'Monthly_Spend_Trend_SGD', 'Seasonal_Spend_Variation', 'Weekend_Spend_Ratio',
                'Festive_Season_Spend_SGD', 'Total_Monthly_Spend_SGD', 'Spend_to_Income_Ratio',
                'UPI_Transaction_Count', 'Debit_Card_Transaction_Count', 'Credit_Card_Transaction_Count',
                'Cash_Withdrawal_Count', 'Recurring_Transaction_Count', 'Recurring_Payment_Ratio',
                'Savings_to_Spend_Ratio', 'Spend_Growth_Rate_YoY', 'High_Value_Transaction_Count',
                'Flight_Risk_Score', 'Financial_Stress_Score', 'AAR_Score', 'Financial_Health_Score',
                'Successful_Contacts_Count', 'Contact_Success_Rate', 'Customer_Best_Agent_Interaction_Count',
                'App_Login_Frequency', 'Online_Banking_Activity', 'Monthly_Income_SGD',
                'No_of_Valid_Numbers', 'No_of_Invalid_Numbers', 'Mobile_Number_Change_Count',
                'Mobile_Number_Change_Count_This_Year', 'Address_Change_Count',
                'Address_Change_Count_This_Year', 'Contact_Data_Change_Frequency']

    def _get_categorical_columns(self):
        """Define categorical columns for preprocessing (Region will be encoded here)"""
        return ['Product_Type', 'Payment_Frequency', 'Settlement_History',
                'Channel_used', 'Response_Outcome',
                'Language_Preference', 'Smartphone_Penetration', 'Preferred_Channel',
                'Call_SMS_Activity_Patterns', 'WhatsApp_OTT_usage_Indicator',
                'Regional_Time_Restrictions', 'Communication_Complaince_Limits',
                'Gender', 'Occupation', 'Employeement_Type', 'Customer_Employment_Status',
                'Finance_Stress_Status', 'Preferred_Payment_Channel', 'Financial_Health_Status',
                'Avg_Balance_Trends', 'AAR_Risk_Level', 'Region', 'Area']

    def _get_agent_columns(self):
        """Define agent columns that should be left as-is"""
        return ['Last_Successful_Agent_ID', 'Best_Contact_Agent_IDs']

    def _get_passthrough_columns(self):
        """Define columns that should be passed through without transformation"""
        return ['Area_From_Address', 'Pincode_From_Address', 'Address', 'Pincode',
                'Customer_id', 'Loan_Account_id', 'Name']

    def _get_date_columns(self):
        """Define date columns for processing"""
        return ['Installment_Due_Date', 'Last_Payment_Date']

    def _get_columns_to_drop(self):
        """Define columns to drop from the dataset"""
        return ['Primary_Phone_Number', 'Secondary_Mobile_Number', 'Landline_Phone_Number']

    def _build_pipeline(self):
        """Build the updated pipeline"""

        # Main preprocessing pipeline
        self.pipeline = Pipeline([
            # Step 1: Load data
            ('data_loader', DataLoader(self.file_path)),

            # Step 2: Drop phone number columns
            ('column_dropper', ColumnDropper(self._get_columns_to_drop())),

            # Step 3: Process address information (extract area and pincode but don't encode)
            ('address_processor', AddressProcessor()),

            # Step 4: Validate email addresses
            ('email_validator', EmailValidator()),

            # Step 5: Convert boolean columns
            ('boolean_converter', BooleanConverter()),

            # Step 6: Encode income bands
            ('income_encoder', IncomeBandEncoder()),

            # Step 7: Extract date features
            ('date_extractor', DateFeatureExtractor(self._get_date_columns())),
        ])

    def fit_transform(self, save_path=None):
        """Run the complete pipeline and return processed data"""
        print("=" * 50)
        print("RUNNING SINGAPORE LOAN DATA PIPELINE")
        print("=" * 50)

        try:
            # Run the pipeline steps manually to maintain DataFrame structure
            df = self.pipeline.named_steps['data_loader'].transform(None)
            original_shape = df.shape

            print("Applying preprocessing steps...")
            for step_name, transformer in list(self.pipeline.named_steps.items())[1:]:
                print(f"Applying {step_name}...")
                df = transformer.fit_transform(df)

            # Remove duplicates
            df = df.drop_duplicates()

            # Apply standard preprocessing to categorical columns only
            # Numeric columns will be handled in feature engineering
            categorical_features = [col for col in self._get_categorical_columns() if col in df.columns]
            agent_features = [col for col in self._get_agent_columns() if col in df.columns]
            passthrough_features = [col for col in self._get_passthrough_columns() if col in df.columns]

            # Handle categorical columns (including Region and Area)
            if categorical_features:
                categorical_imputer = SimpleImputer(strategy='most_frequent')
                onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

                # Impute missing values
                df[categorical_features] = categorical_imputer.fit_transform(df[categorical_features])

                # One-hot encode categorical variables (including Region and Area)
                encoded_array = onehot_encoder.fit_transform(df[categorical_features])
                encoded_columns = onehot_encoder.get_feature_names_out(categorical_features)

                # Create DataFrame for encoded features
                encoded_df = pd.DataFrame(encoded_array, columns=encoded_columns, index=df.index)

                # Drop original categorical columns and add encoded ones
                df = df.drop(columns=categorical_features)
                df = pd.concat([df, encoded_df], axis=1)

            # Agent columns, Area_From_Address, Pincode_From_Address, and other passthrough columns are left as-is
            # They remain in the DataFrame

            self.processed_data = df.reset_index(drop=True)

            # Save if path provided
            if save_path:
                self.save_processed_data(save_path)

            print("Pipeline completed successfully!")
            print(f"Original shape: {original_shape}")
            print(f"Processed shape: {self.processed_data.shape}")

            return self.processed_data

        except Exception as e:
            print(f"Error in pipeline: {e}")
            import traceback
            traceback.print_exc()
            return None

    def save_processed_data(self, path):
        """Save the processed data"""
        if self.processed_data is not None:
            self.processed_data.to_csv(path, index=False)
            print(f"Processed data saved to {path}")

            # Also save the pipeline for future use
            joblib.dump(self.pipeline, 'data_pipeline.pkl')
            print("Pipeline saved as 'data_pipeline.pkl'")

    def load_and_transform_new_data(self, new_data_path):
        """Load and transform new data using the fitted pipeline"""
        if self.pipeline is None:
            raise ValueError("Pipeline not fitted yet. Call fit_transform first.")

        # Load new data
        new_data_loader = DataLoader(new_data_path)
        new_df = new_data_loader.transform(None)

        # Apply the same transformations
        for step_name, transformer in list(self.pipeline.named_steps.items())[1:]:
            new_df = transformer.transform(new_df)

        return new_df

In [3]:

# Initialize the Singapore loan data pipeline
pipeline = SingaporeLoanDataPipeline(r'/content/drive/MyDrive/Project/singapore_loan_data.csv')

# Run the complete pipeline
processed_data = pipeline.fit_transform(save_path='processed_loan_data.csv')

if processed_data is not None:
    # Display results
    print("\nPipeline Summary:")
    print(f"Processed data shape: {processed_data.shape}")
    print(f"Processed data columns: {len(processed_data.columns)}")

    print("\nSample of processed data:")
    display(processed_data.head())

    print("\nData types after processing:")
    print(processed_data.dtypes.value_counts())
else:
    print("Pipeline failed to process data.")


RUNNING SINGAPORE LOAN DATA PIPELINE
Loading data...
Successfully loaded 100,000 rows
Applying preprocessing steps...
Applying column_dropper...
Applying address_processor...
Applying email_validator...
Applying boolean_converter...
Applying income_encoder...
Applying date_extractor...
Processed data saved to processed_loan_data.csv
Pipeline saved as 'data_pipeline.pkl'
Pipeline completed successfully!
Original shape: (100000, 106)
Processed shape: (100000, 192)

Pipeline Summary:
Processed data shape: (100000, 192)
Processed data columns: 192

Sample of processed data:


,Customer_id,Loan_Account_id,Loan_Amount_SGD,Outstanding_Balance_SGD,Day_Past_Due,Tenure,Interest_Rate,Current_EMI_SGD,Partial_Payment_Indicator,Number_of_Past_Payments,...,Area_Raffles Boulevard,Area_Raffles Place,Area_Scotts Road,Area_Sentosa Gateway,Area_Serangoon Garden Circus,Area_Serangoon Garden Way,Area_Suntec City Drive,Area_Tanglin Road,Area_Telok Blangah Road,Area_Temasek Avenue
0,SCB909522509,20194238,100000.0,88406.56,0,24,8.5,4545.57,False,3,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,SCB982519764,94084974,50000.0,30420.52,3,24,8.5,2318.24,False,11,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,SCB934804341,63516731,20000.0,7021.52,6,60,9.0,423.47,False,43,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,SCB929520634,61907476,39000.0,38222.97,11,60,8.5,816.14,False,3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,SCB930205962,10161688,100000.0,96023.19,0,24,9.0,4568.47,False,7,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0



Data types after processing:
float64    128
int64       35
bool        13
object      10
int32        6
Name: count, dtype: int64


In [4]:

import pandas as pd
import numpy as np

def reconstruct_categorical_features(df, categorical_prefixes):
    """
    Reconstruct original categorical columns from One-Hot Encoded columns.
    Assumes OHE columns are named 'Prefix_Value'.
    """
    df_reconstructed = df.copy()

    for prefix in categorical_prefixes:
        # Find all columns starting with this prefix
        # We assume the format is Prefix_Value
        relevant_cols = [col for col in df.columns if col.startswith(f"{prefix}_")]

        if not relevant_cols:
            continue

        # Create a new column with the prefix name
        # We use idxmax to find the column with value 1, then strip the prefix
        # This assumes mutually exclusive categories (which OHE implies)

        # Check if we have the original column already (unlikely if fully processed)
        if prefix in df.columns:
            continue

        def get_category(row):
            for col in relevant_cols:
                if row[col] == 1:
                    return col[len(prefix)+1:] # Remove 'Prefix_'
            return "Unknown"

        df_reconstructed[prefix] = df[relevant_cols].apply(get_category, axis=1)

    return df_reconstructed

# User-provided logic functions
def calculate_aar_score(upi_count, debit_count, credit_count, cash_count, recurring_count,
                       total_transactions, monthly_income, total_spend, age, occupation):
    """Calculate AAR score based on transaction patterns"""

    # Base factors
    transaction_diversity = min(1.0, (upi_count + debit_count + credit_count) / max(1, total_transactions) * 2)
    recurring_ratio = recurring_count / max(1, total_transactions)
    digital_adoption = (upi_count + debit_count + credit_count) / max(1, total_transactions)

    # Age factor
    if age < 30:
        age_factor = 1.1
    elif age > 60:
        age_factor = 0.9
    else:
        age_factor = 1.0

    # Occupation factor
    if occupation in ["Employed", "Self-Employed"]:
        occupation_factor = 1.05
    elif occupation == "Student":
        occupation_factor = 0.95
    else:
        occupation_factor = 1.0

    # Calculate base AAR score
    base_score = (
        transaction_diversity * 0.4 +
        recurring_ratio * 0.3 +
        digital_adoption * 0.3
    ) * 0.7  # Scale to 0.7 max

    # Apply factors
    aar_score = base_score * age_factor * occupation_factor

    # Clip score to a reasonable 0-1 range
    aar_score = np.clip(aar_score, 0, 1)

    # Round to 6 decimal places
    aar_score = round(aar_score, 6)

    # Risk level classification
    if aar_score <= 0.5:
        aar_risk = "Low"
    elif aar_score <= 0.65:
        aar_risk = "Medium"
    else:
        aar_risk = "High"

    return aar_score, aar_risk

def calculate_financial_stress_score(monthly_income, total_spend, savings_ratio, payment_history,
                                   missed_payments, employment_status, age, occupation, debt_to_income_ratio):
    """Calculate realistic financial stress score"""

    # Factor 1: Spending to Income Ratio (30% weight)
    spend_income_ratio = total_spend / monthly_income if monthly_income > 0 else 2.0
    if spend_income_ratio <= 0.6:
        factor1 = 0.1
    elif spend_income_ratio <= 0.8:
        factor1 = 0.3
    elif spend_income_ratio <= 1.0:
        factor1 = 0.6
    elif spend_income_ratio <= 1.2:
        factor1 = 0.8
    else:
        factor1 = 1.0

    # Factor 2: Savings Ratio (25% weight)
    if savings_ratio >= 0.2:
        factor2 = 0.1
    elif savings_ratio >= 0.1:
        factor2 = 0.3
    elif savings_ratio >= 0:
        factor2 = 0.6
    elif savings_ratio >= -0.1:
        factor2 = 0.8
    else:
        factor2 = 1.0

    # Factor 3: Payment History (20% weight)
    factor3 = 1 - payment_history

    # Factor 4: Missed Payments (15% weight)
    missed_payment_factor = min(1.0, missed_payments * 0.2)

    # Factor 5: Debt-to-Income Ratio (10% weight)
    if debt_to_income_ratio <= 0.3:
        factor5 = 0.1
    elif debt_to_income_ratio <= 0.5:
        factor5 = 0.3
    elif debt_to_income_ratio <= 0.7:
        factor5 = 0.6
    elif debt_to_income_ratio <= 1.0:
        factor5 = 0.8
    else:
        factor5 = 1.0

    # Calculate weighted stress score
    weighted_score = (
        factor1 * 0.30 +
        factor2 * 0.25 +
        factor3 * 0.20 +
        missed_payment_factor * 0.15 +
        factor5 * 0.10
    )

    # Age and employment adjustments
    if employment_status == "Unemployed":
        weighted_score = min(1.0, weighted_score * 1.3)
    elif occupation == "Student":
        weighted_score = min(1.0, weighted_score * 0.9)
    elif age > 60:
        weighted_score = min(1.0, weighted_score * 1.2)

    financial_stress_score = weighted_score * 100

    # Stress status classification
    if financial_stress_score <= 25:
        stress_status = "Low stress"
    elif financial_stress_score <= 50:
        stress_status = "Medium stress"
    elif financial_stress_score <= 75:
        stress_status = "High stress"
    else:
        stress_status = "Extreme High stress"

    return stress_status, round(financial_stress_score, 2)

def calculate_financial_health_status(financial_stress_score, balance_trend, savings_ratio,
                                    employment_status, overdraft_flag):
    """Calculate holistic Financial Health Status"""
    health_score = 0

    # 1. Factor in the Stress Status
    if financial_stress_score <= 25:
        health_score += 2
    elif financial_stress_score <= 50:
        health_score += 0
    elif financial_stress_score <= 75:
        health_score -= 1
    else:
        health_score -= 2

    # 2. Factor in Balance Trends
    if balance_trend == "Rising":
        health_score += 2
    elif balance_trend == "Stable":
        health_score += 1
    elif balance_trend == "Falling":
        health_score -= 2

    # 3. Factor in Savings Ratio
    if savings_ratio > 0.1:  # Good saver
        health_score += 1
    elif savings_ratio < 0:  # Spending savings or going into debt
        health_score -= 1

    # 4. Factor in Employment
    if employment_status == "Unemployed":
        health_score -= 1

    # 5. Factor in Overdrafts
    if overdraft_flag:
        health_score -= 1

    # Map the final health_score to the Health Status
    if health_score >= 3:
        health_status = "Healthy"
    elif health_score > 0:  # Score of 1 or 2
        health_status = "Moderate"
    elif health_score > -3: # Score of 0, -1, or -2
        health_status = "Stressed"
    else: # Score of -3 or less
        health_status = "Critical"

    return health_status, health_score

def calculate_spend_to_income_ratio(total_monthly_spend, monthly_income):
    """Calculate spend to income ratio"""
    if monthly_income > 0:
        spend_ratio = total_monthly_spend / monthly_income
    else:
        spend_ratio = 0

    return round(spend_ratio, 2)

def calculate_digital_savviness_score(upi_count, debit_count, credit_count, cash_count,
                                    recurring_count, total_transactions, app_login_frequency,
                                    online_banking_activity, smartphone_penetration,
                                    whatsapp_ott_usage, age, occupation, preferred_channel):
    """Calculate digital savviness score"""

    # 1. Digital Transaction Ratio (30% weight)
    digital_transactions = upi_count + debit_count + credit_count
    total_non_zero = max(1, total_transactions)
    digital_ratio = digital_transactions / total_non_zero

    # 2. Recurring Transaction Ratio (20% weight)
    recurring_ratio = recurring_count / total_non_zero

    # 3. App Usage Factor (20% weight)
    app_usage_factor = min(1.0, app_login_frequency / 30)

    # 4. Online Banking Activity (15% weight)
    online_banking_factor = min(1.0, online_banking_activity / 25)

    # 5. Digital Channel Preference (15% weight)
    channel_scores = {
        "WhatsApp": 1.0, "Email": 0.8, "SMS": 0.6,
        "Call": 0.4, "Field Agent": 0.2, "IVR": 0.3
    }
    channel_score = channel_scores.get(preferred_channel, 0.5)

    # Calculate base digital savviness score
    base_score = (
        digital_ratio * 0.30 +
        recurring_ratio * 0.20 +
        app_usage_factor * 0.20 +
        online_banking_factor * 0.15 +
        channel_score * 0.15
    )

    # Apply demographic and technology factors
    if age < 25: age_factor = 1.2
    elif age < 35: age_factor = 1.1
    elif age < 50: age_factor = 1.0
    elif age < 60: age_factor = 0.9
    else: age_factor = 0.8

    occupation_factors = {
        "Student": 1.3, "Employed": 1.1, "Self-Employed": 1.2,
        "Homemaker": 0.9, "Retired": 0.8, "Unemployed": 0.7
    }
    occupation_factor = occupation_factors.get(occupation, 1.0)

    smartphone_factors = {"High": 1.2, "Medium": 1.0, "Low": 0.7}
    smartphone_factor = smartphone_factors.get(smartphone_penetration, 1.0)

    whatsapp_factor = 1.1 if whatsapp_ott_usage else 0.9

    adjusted_score = base_score * age_factor * occupation_factor * smartphone_factor * whatsapp_factor
    final_score = max(0, min(1, adjusted_score))

    if final_score >= 0.8: savviness_level = "Highly Digital"
    elif final_score >= 0.6: savviness_level = "Digital Adopter"
    elif final_score >= 0.4: savviness_level = "Moderate Digital"
    elif final_score >= 0.2: savviness_level = "Low Digital"
    else: savviness_level = "Traditional"

    return round(final_score, 4), savviness_level, round(digital_ratio, 4), round(recurring_ratio, 4), round(app_usage_factor, 4), round(online_banking_factor, 4), round(channel_score, 4)



def calculate_flight_risk_score(financial_health_score, financial_stress_score,
                              address_change_count, address_change_count_this_year,
                              mobile_change_count, mobile_change_count_this_year,
                              invalid_mobile_count, valid_mobile_count,
                              email_valid_format):
    """
    Calculate flight risk score based on financial health, stress, and contact stability.

    Parameters:
    - financial_health_score: 0-100 score (Higher is better)
    - financial_stress_score: 0-100 score (Higher is worse)
    - address_change_count: Total address changes
    - address_change_count_this_year: Address changes in current year
    - mobile_change_count: Total mobile number changes
    - mobile_change_count_this_year: Mobile changes in current year
    - invalid_mobile_count: Count of invalid numbers
    - valid_mobile_count: Count of valid numbers
    - email_valid_format: Boolean/Binary indicating valid email format

    Returns:
    - flight_risk_score: Rounded score between 0-1
    """

    # Normalize scores to 0-1
    norm_health = financial_health_score / 100.0 if financial_health_score is not None else 0.5
    norm_stress = financial_stress_score / 100.0 if financial_stress_score is not None else 0.5

    # Base Risk from Financials (40%)
    # High Stress -> High Risk, Low Health -> High Risk
    financial_risk = (norm_stress * 0.6) + ((1 - norm_health) * 0.4)

    # Stability Risk from Changes (40%)
    # Address Changes
    addr_risk = 0.0
    if address_change_count is not None:
        addr_risk += min(0.5, address_change_count * 0.1)
    if address_change_count_this_year is not None:
        addr_risk += min(0.5, address_change_count_this_year * 0.2) # Recent changes are riskier

    # Mobile Changes
    mobile_risk = 0.0
    if mobile_change_count is not None:
        mobile_risk += min(0.5, mobile_change_count * 0.1)
    if mobile_change_count_this_year is not None:
        mobile_risk += min(0.5, mobile_change_count_this_year * 0.2)

    stability_risk = min(1.0, addr_risk + mobile_risk)

    # Data Quality/Validity Risk (20%)
    data_risk = 0.0
    if invalid_mobile_count is not None and invalid_mobile_count > 0:
        data_risk += 0.3
    if valid_mobile_count is not None and valid_mobile_count == 0:
        data_risk += 0.4
    if email_valid_format is not None:
        # Check if it's False or 0
        if email_valid_format == 0 or email_valid_format == False or str(email_valid_format).lower() == 'false':
            data_risk += 0.3

    data_risk = min(1.0, data_risk)

    # Total Weighted Score
    total_risk = (financial_risk * 0.4) + (stability_risk * 0.4) + (data_risk * 0.2)

    # Add some randomness for distribution
    total_risk += np.random.uniform(-0.05, 0.05)

    return round(max(0.0, min(1.0, total_risk)), 2)
def calculate_flight_risk_simple(payment_history, missed_payments, employment_status, age):
    """
    Simplified flight risk calculation for basic implementation
    """
    # Behavioral factors (40%)
    behavioral_factors = (1 - payment_history) * 0.4

    # Stability factors (35%)
    missed_payment_factor = min(1.0, missed_payments * 0.15) * 0.25
    employment_factor = 0.0
    if employment_status == "Unemployed":
        employment_factor = 0.08
    elif employment_status == "Student":
        employment_factor = 0.05
    stability_factors = missed_payment_factor + employment_factor

    # Demographic factors (25%)
    demographic_factors = 0.0
    if age < 25:
        demographic_factors = 0.15
    elif age > 60:
        demographic_factors = 0.05
    else:
        demographic_factors = 0.08

    # Calculate total
    flight_risk = behavioral_factors + stability_factors + demographic_factors

    # Age adjustments
    if age < 30:
        flight_risk = min(1.0, flight_risk * 1.2)
    elif age > 60:
        flight_risk = min(1.0, flight_risk * 0.8)

    # Random variation for realism
    flight_risk += np.random.uniform(-0.05, 0.05)
    flight_risk = max(0, min(1.0, flight_risk))

    # Reduced extreme cases
    if np.random.random() < 0.02:
        flight_risk = np.random.choice([
            np.random.uniform(0.7, 0.9),
            np.random.uniform(0.05, 0.15)
        ])

    return round(flight_risk, 2)

def get_flight_risk_category(flight_risk_score):
    """
    Convert flight risk score to category
    """
    if flight_risk_score <= 0.3:
        return "Low Risk"
    elif flight_risk_score <= 0.6:
        return "Medium Risk"
    elif flight_risk_score <= 0.8:
        return "High Risk"
    else:
        return "Very High Risk"
# Main Feature Engineering Execution
print("Loading processed data...")
try:
    df = pd.read_csv('processed_loan_data.csv')
    print(f"Loaded {len(df)} rows.")

    # 1. Reconstruct Categorical Features needed for logic
    print("Reconstructing categorical features...")
    categorical_prefixes = [
        'Occupation', 'Employeement_Type', 'Preferred_Channel',
        'Smartphone_Penetration', 'Avg_Balance_Trends', 'Region',
        'WhatsApp_OTT_usage_Indicator'
    ]
    df_temp = reconstruct_categorical_features(df, categorical_prefixes)

    # 2. Apply Feature Engineering
    print("Applying feature engineering logic...")

    # Pre-calculate common values
    df_temp['Total_Transactions'] = (
        df_temp['UPI_Transaction_Count'] +
        df_temp['Debit_Card_Transaction_Count'] +
        df_temp['Credit_Card_Transaction_Count'] +
        df_temp['Cash_Withdrawal_Count']
    )

    # AAR Score
    print("Calculating AAR Score...")
    aar_results = df_temp.apply(lambda row: calculate_aar_score(
        row['UPI_Transaction_Count'], row['Debit_Card_Transaction_Count'],
        row['Credit_Card_Transaction_Count'], row['Cash_Withdrawal_Count'],
        row['Recurring_Transaction_Count'], row['Total_Transactions'],
        row['Monthly_Income_SGD'], row['Total_Monthly_Spend_SGD'],
        row['Age'], row['Occupation']
    ), axis=1)
    df_temp['aar_score'] = [x[0] for x in aar_results]
    df_temp['customer_risk_level'] = [x[1] for x in aar_results]

    # Financial Stress
    print("Calculating Financial Stress...")
    # Derive inputs
    df_temp['debt_to_income_ratio'] = np.where(df_temp['Monthly_Income_SGD'] > 0,
                                              df_temp['Current_EMI_SGD'] / df_temp['Monthly_Income_SGD'], 0)
    # Assuming payment_history is related to missed payments or delinquency
    # Using 1 if no missed payments, 0 otherwise as a simple proxy if not available
    df_temp['payment_history_proxy'] = np.where(df_temp['Missed_Payments_Count'] == 0, 1.0, 0.0)

    stress_results = df_temp.apply(lambda row: calculate_financial_stress_score(
        row['Monthly_Income_SGD'], row['Total_Monthly_Spend_SGD'],
        row['Savings_to_Spend_Ratio'], row['payment_history_proxy'],
        row['Missed_Payments_Count'], row['Employeement_Type'],
        row['Age'], row['Occupation'], row['debt_to_income_ratio']
    ), axis=1)
    df_temp['finance_stress_status'] = [x[0] for x in stress_results]
    df_temp['financial_stress_score'] = [x[1] for x in stress_results]

    # Financial Health
    print("Calculating Financial Health...")
    health_results = df_temp.apply(lambda row: calculate_financial_health_status(
        row['financial_stress_score'], row['Avg_Balance_Trends'],
        row['Savings_to_Spend_Ratio'], row['Employeement_Type'],
        row['Overdraft_or_Low_Balance_Flag']
    ), axis=1)
    df_temp['financial_health_status'] = [x[0] for x in health_results]
    df_temp['financial_health_score'] = [x[1] for x in health_results]

    # Spend Ratio
    df_temp['spend_to_income_ratio'] = df_temp.apply(lambda row: calculate_spend_to_income_ratio(
        row['Total_Monthly_Spend_SGD'], row['Monthly_Income_SGD']
    ), axis=1)

    # Digital Savviness
    print("Calculating Digital Savviness...")
    savviness_results = df_temp.apply(lambda row: calculate_digital_savviness_score(
        row['UPI_Transaction_Count'], row['Debit_Card_Transaction_Count'],
        row['Credit_Card_Transaction_Count'], row['Cash_Withdrawal_Count'],
        row['Recurring_Transaction_Count'], row['Total_Transactions'],
        row['App_Login_Frequency'], row['Online_Banking_Activity'],
        row['Smartphone_Penetration'], row['WhatsApp_OTT_usage_Indicator'],
        row['Age'], row['Occupation'], row['Preferred_Channel']
    ), axis=1)

    df_temp['digital_savviness_score'] = [x[0] for x in savviness_results]
    df_temp['digital_savviness_level'] = [x[1] for x in savviness_results]
    df_temp['digital_transaction_ratio'] = [x[2] for x in savviness_results]
    df_temp['recurring_transaction_ratio'] = [x[3] for x in savviness_results]
    df_temp['app_usage_factor'] = [x[4] for x in savviness_results]
    df_temp['online_banking_factor'] = [x[5] for x in savviness_results]
    df_temp['channel_preference_score'] = [x[6] for x in savviness_results]

    # Additional Channel Optimization Features
    print("Calculating Additional Features...")

    # Loan: EMI to Income Ratio (Affordability)
    df_temp['EMI_to_Income_Ratio'] = np.where(df_temp['Monthly_Income_SGD'] > 0,
                                             df_temp['Current_EMI_SGD'] / df_temp['Monthly_Income_SGD'], 0)

    # Loan: Credit Utilization Proxy
    df_temp['Credit_Utilization_Proxy'] = np.where(df_temp['Loan_Amount_SGD'] > 0,
                                                  df_temp['Outstanding_Balance_SGD'] / df_temp['Loan_Amount_SGD'], 0)

    # Financial: Income Stability Score (Proxy)
    # Higher for employed/retired, lower for unemployed/student
    stability_map = {'Employed': 1.0, 'Retired': 0.9, 'Self-Employed': 0.8, 'Homemaker': 0.7, 'Student': 0.5, 'Unemployed': 0.3}
    df_temp['Income_Stability_Score'] = df_temp['Employeement_Type'].map(stability_map).fillna(0.5)

    # Spend: Essential vs Non-Essential Ratio
    # Essential: Utility, Health, Education, Travel(Commute)
    # Non-Essential: Shopping, Entertainment
    essential_spend = df_temp['Utility_Spend_SGD'] + df_temp['Health_Spend_SGD'] + df_temp['Education_Spend_SGD']
    non_essential_spend = df_temp['Shopping_Spend_SGD'] + df_temp['Entertainment_Spend_SGD']
    df_temp['Essential_vs_NonEssential_Ratio'] = np.where(non_essential_spend > 0,
                                                         essential_spend / non_essential_spend, 10.0) # Cap at 10 if 0 non-essential

    # Communication: Agent Stickiness
    # If Best_Contact_Agent_IDs is present, it implies stickiness.
    # We can use Customer_Best_Agent_Interaction_Count / Successful_Contacts_Count
    df_temp['Agent_Stickiness'] = np.where(df_temp['Successful_Contacts_Count'] > 0,
                                          df_temp['Customer_Best_Agent_Interaction_Count'] / df_temp['Successful_Contacts_Count'], 0)

    # Communication: Contact Fatigue Index
    # No_of_Attempts / (Successful_Contacts_Count + 1)
    df_temp['Contact_Fatigue_Index'] = df_temp['No_of_Attempts'] / (df_temp['Successful_Contacts_Count'] + 1)

    # Geographical: Area Income Percentile (Mock logic as we don't have external census data)
    # We can calculate the average income per Area from our dataset and assign a percentile
    area_income = df_temp.groupby('Area_From_Address')['Monthly_Income_SGD'].transform('mean')
    df_temp['Area_Income_Percentile'] = df_temp['Monthly_Income_SGD'] / area_income

    # DEBUG: Check for Credit_Utilization_Proxy
    if 'Credit_Utilization_Proxy' not in df_temp.columns:
        print("CRITICAL ERROR: Credit_Utilization_Proxy is MISSING from df_temp columns!")
        print(f"Available columns: {list(df_temp.columns)}")
    else:
        print("DEBUG: Credit_Utilization_Proxy found. Proceeding with Flight Risk calculation.")

    # Flight Risk Score
    print("Calculating Flight Risk...")
    flight_risk_results = df_temp.apply(lambda row: calculate_flight_risk_score(
        row['financial_health_score'], row['financial_stress_score'],
        row.get('Address_Change_Count', 0), row.get('Address_Change_Count_This_Year', 0),
        row.get('Mobile_Number_Change_Count', 0), row.get('Mobile_Number_Change_Count_This_Year', 0),
        row.get('No_of_Invalid_Numbers', 0), row.get('No_of_Valid_Numbers', 0),
        row.get('Email_Valid_Format', 1)
    ), axis=1)
    df_temp['flight_risk_score'] = flight_risk_results
    df_temp['flight_risk_category'] = df_temp['flight_risk_score'].apply(get_flight_risk_category)

    # 3. Filter Columns for Final Output
    print("Filtering columns...")

    # Identify columns used as inputs (to be excluded if they are original raw features)
    # We want to keep IDs and the NEW features.

    new_features = [
        'aar_score', 'customer_risk_level',
        'finance_stress_status', 'financial_stress_score',
        'financial_health_status', 'financial_health_score',
        'spend_to_income_ratio',
        'digital_savviness_score', 'digital_savviness_level',
        'digital_transaction_ratio', 'recurring_transaction_ratio',
        'app_usage_factor', 'online_banking_factor', 'channel_preference_score',
        'EMI_to_Income_Ratio', 'Credit_Utilization_Proxy',
        'Income_Stability_Score', 'Essential_vs_NonEssential_Ratio',
        'Agent_Stickiness', 'Contact_Fatigue_Index', 'Area_Income_Percentile'
    ]

    input_columns_to_drop = [
        'UPI_Transaction_Count', 'Debit_Card_Transaction_Count', 'Credit_Card_Transaction_Count',
        'Cash_Withdrawal_Count', 'Recurring_Transaction_Count', 'Total_Transactions',
        'Monthly_Income_SGD', 'Total_Monthly_Spend_SGD', 'Age',
        'Current_EMI_SGD', 'Missed_Payments_Count', 'Savings_to_Spend_Ratio',
        'Overdraft_or_Low_Balance_Flag', 'App_Login_Frequency', 'Online_Banking_Activity',
        'WhatsApp_OTT_usage_Indicator', 'Utility_Spend_SGD', 'Health_Spend_SGD',
        'Education_Spend_SGD', 'Shopping_Spend_SGD', 'Entertainment_Spend_SGD',
        'Successful_Contacts_Count', 'Customer_Best_Agent_Interaction_Count', 'No_of_Attempts',
        'debt_to_income_ratio', 'payment_history_proxy', # Intermediate cols
        'Address', # Removed as requested
        'Contact_Success_Rate', 'Mobile_Number_Change_Count', 'Address_Change_Count', # Old Inputs
        'Address_Change_Count_This_Year', 'Mobile_Number_Change_Count_This_Year',
        'No_of_Invalid_Numbers', 'No_of_Valid_Numbers', 'Email_Valid_Format' # New Inputs
    ]

    # Also drop the reconstructed categorical columns as they were temporary
    input_columns_to_drop.extend(categorical_prefixes)

    # Keep ID columns
    id_columns = ['Customer_id', 'Loan_Account_id']

    # Get all columns from processed data that are NOT in the drop list
    # Note: processed_loan_data.csv has OHE columns (e.g. Occupation_Student).
    # The user asked to drop "features that are not used".
    # Strictly speaking, 'Occupation_Student' was used (via reconstruction).
    # So we should probably drop the OHE columns corresponding to the used categoricals too.

    cols_to_keep = []
    for col in df.columns:
        # Check if this column belongs to a used categorical prefix
        is_used_categorical = False
        for prefix in categorical_prefixes:
            if col.startswith(f"{prefix}_"):
                is_used_categorical = True
                break

        if col not in input_columns_to_drop and not is_used_categorical:
            cols_to_keep.append(col)

    # Combine kept original columns + new features
    # Enforce order: IDs first, then new features, then remaining original columns

    # 1. IDs
    ordered_columns = ['Customer_id', 'Loan_Account_id']

    # 2. New Features (already defined in new_features list)
    ordered_columns.extend([c for c in new_features if c not in ordered_columns])

    # 3. Remaining Original Columns (cols_to_keep)
    # Sort them alphabetically for better readability
    remaining_cols = [c for c in cols_to_keep if c not in ordered_columns]
    remaining_cols.sort()
    ordered_columns.extend(remaining_cols)

    final_columns = ordered_columns

    # Create final dataframe
    df_final = df_temp[final_columns].copy()

    # Save
    print("Saving feature engineered data...")
    output_file = 'feature_engineered.csv'
    try:
        df_final.to_csv(output_file, index=False)
        print(f"Saved {len(df_final)} rows to '{output_file}'")
    except PermissionError:
        print(f"Warning: Could not write to '{output_file}' (File might be open).")
        output_file = 'feature_engineered_v2.csv'
        print(f"Attempting to save to '{output_file}' instead...")
        df_final.to_csv(output_file, index=False)
        print(f"Saved {len(df_final)} rows to '{output_file}'")

    print(f"Final column count: {len(df_final.columns)}")

    # Display sample
    display(df_final.head())

except Exception as e:
    print(f"Error in feature engineering: {e}")
    import traceback
    traceback.print_exc()


Loading processed data...
Loaded 100000 rows.
Reconstructing categorical features...
Applying feature engineering logic...
Calculating AAR Score...
Calculating Financial Stress...
Calculating Financial Health...
Calculating Digital Savviness...
Calculating Additional Features...
DEBUG: Credit_Utilization_Proxy found. Proceeding with Flight Risk calculation.
Calculating Flight Risk...
Filtering columns...
Saving feature engineered data...
Saved 100000 rows to 'feature_engineered.csv'
Final column count: 150


,Customer_id,Loan_Account_id,aar_score,customer_risk_level,finance_stress_status,financial_stress_score,financial_health_status,financial_health_score,spend_to_income_ratio,digital_savviness_score,...,Settlement_History_Not Settled,Settlement_History_Partial Settlement,Settlement_History_Settled,Settlement_History_Under Negotiation,Spend_Growth_Rate_YoY,Tenure,Travel_Spend_SGD,Unemployeement_rate_region,Valid_Phone_Number,Weekend_Spend_Ratio
0,SCB909522509,20194238,0.580218,Medium,Medium stress,32.00,Moderate,1,0.63,1.0000,...,0.0,0.0,1.0,0.0,0.25,24,919.57,1.77,True,0.40
1,SCB982519764,94084974,0.558600,Medium,Medium stress,41.50,Moderate,2,0.54,0.5953,...,0.0,0.0,0.0,1.0,0.07,24,341.63,1.82,True,0.33
2,SCB934804341,63516731,0.561273,Medium,Medium stress,40.50,Healthy,3,0.76,0.5411,...,0.0,1.0,0.0,0.0,0.05,60,518.99,1.87,True,0.34
3,SCB929520634,61907476,0.597188,Medium,Medium stress,36.50,Moderate,2,0.57,0.9708,...,1.0,0.0,0.0,0.0,0.13,60,278.40,1.88,True,0.34
4,SCB930205962,10161688,0.463050,Low,High stress,63.05,Moderate,1,0.55,0.3879,...,0.0,0.0,0.0,1.0,0.09,24,652.91,1.90,True,0.40


In [5]:
# Cell to drop specific columns as requested and update the CSV
try:
    print("Loading feature_engineered.csv for cleanup...")
    # Load the file (handling potential v2 if v1 is locked, though user asked for v1)
    filename = 'feature_engineered.csv'
    try:
        df_final = pd.read_csv(filename)
    except FileNotFoundError:
        filename = 'feature_engineered.csv'
        print(f"'{filename}' not found, trying '{filename}'...")
        df_final = pd.read_csv(filename)

    print(f"Loaded {len(df_final)} rows from {filename}")

    cols_to_drop_request = [
        'Loan_Account_id',
        'Area_Amber Road', 'Area_Battery Road', 'Area_Bayfront Avenue',
        'Area_Bukit Timah Road', 'Area_Dunearn Road', 'Area_East Coast Road',
        'Area_HarbourFront Walk', 'Area_Holland Road', 'Area_Kensington Park Road',
        'Area_Marina Boulevard', 'Area_Marine Parade Road', 'Area_Market Street',
        'Area_Orchard Road', 'Area_Raffles Boulevard', 'Area_Raffles Place',
        'Area_Scotts Road', 'Area_Sentosa Gateway', 'Area_Serangoon Garden Circus',
        'Area_Serangoon Garden Way', 'Area_Suntec City Drive', 'Area_Tanglin Road',
        'Area_Telok Blangah Road', 'Area_Temasek Avenue',
        'Customer_Employment_Status_Employed', 'Customer_Employment_Status_Unemployed',
        'Day_Past_Due', 'Email_Activity', 'Email_Disposable', 'Email_Domain',
        'Email_Domain_Legitimate', 'Email_ID', 'Landline_Phone_Number_Valid',
        'Number_of_Past_Payments', 'Outstanding_Balance_SGD', 'Pincode_From_Address',
        'Secondary_Mobile_Number_Valid', 'Tenure', 'Valid_Phone_Number'
    ]

    # Drop only columns that exist to avoid errors if run multiple times
    existing_cols_to_drop = [c for c in cols_to_drop_request if c in df_final.columns]

    if existing_cols_to_drop:
        print(f"Dropping {len(existing_cols_to_drop)} columns...")
        df_final.drop(columns=existing_cols_to_drop, inplace=True)

        output_filename = 'engineered_features.csv'
        print(f"Saving cleaned data to {output_filename}...")
        df_final.to_csv(output_filename, index=False)
        print("Cleanup complete.")
    else:
        print("No columns to drop found (already dropped?).")

    print(f"Final column count: {len(df_final.columns)}")
    display(df_final.head())

except Exception as e:
    print(f"Error during cleanup: {e}")

Loading feature_engineered.csv for cleanup...
Loaded 100000 rows from feature_engineered.csv
Dropping 39 columns...
Saving cleaned data to engineered_features.csv...
Cleanup complete.
Final column count: 111


,Customer_id,aar_score,customer_risk_level,finance_stress_status,financial_stress_score,financial_health_status,financial_health_score,spend_to_income_ratio,digital_savviness_score,digital_savviness_level,...,Response_Outcome_Promised to pay,Seasonal_Spend_Variation,Settlement_History_Not Settled,Settlement_History_Partial Settlement,Settlement_History_Settled,Settlement_History_Under Negotiation,Spend_Growth_Rate_YoY,Travel_Spend_SGD,Unemployeement_rate_region,Weekend_Spend_Ratio
0,SCB909522509,0.580218,Medium,Medium stress,32.00,Moderate,1,0.63,1.0000,Highly Digital,...,0.0,0.26,0.0,0.0,1.0,0.0,0.25,919.57,1.77,0.40
1,SCB982519764,0.558600,Medium,Medium stress,41.50,Moderate,2,0.54,0.5953,Moderate Digital,...,0.0,0.18,0.0,0.0,0.0,1.0,0.07,341.63,1.82,0.33
2,SCB934804341,0.561273,Medium,Medium stress,40.50,Healthy,3,0.76,0.5411,Moderate Digital,...,0.0,0.30,0.0,1.0,0.0,0.0,0.05,518.99,1.87,0.34
3,SCB929520634,0.597188,Medium,Medium stress,36.50,Moderate,2,0.57,0.9708,Highly Digital,...,0.0,0.15,1.0,0.0,0.0,0.0,0.13,278.40,1.88,0.34
4,SCB930205962,0.463050,Low,High stress,63.05,Moderate,1,0.55,0.3879,Low Digital,...,0.0,0.33,0.0,0.0,0.0,1.0,0.09,652.91,1.90,0.40


In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.base import BaseEstimator, TransformerMixin
import warnings

warnings.filterwarnings('ignore')

class ComprehensiveFinancialProfileCalculator(BaseEstimator, TransformerMixin):
    """Calculate comprehensive financial profile metrics that influence channel preferences"""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # 1. Risk and Health Profile
        if 'aar_score' in X.columns:
            X['AAR_Risk_Level'] = pd.cut(X['aar_score'],
                                        bins=[0, 0.3, 0.5, 0.7, 1.0],
                                        labels=['Very_Low', 'Low', 'Medium', 'High'])

        if 'financial_stress_score' in X.columns:
            X['Financial_Stress_Level'] = pd.cut(X['financial_stress_score'],
                                               bins=[0, 25, 50, 75, 100],
                                               labels=['Low', 'Medium', 'High', 'Very_High'])

        if 'financial_health_score' in X.columns:
            X['Financial_Health_Level'] = pd.cut(X['financial_health_score'],
                                               bins=[-3, 0, 2, 4, 6],
                                               labels=['Critical', 'Stressed', 'Moderate', 'Healthy'])

        # 2. Digital Behavior Profile
        if 'digital_savviness_score' in X.columns:
            X['Digital_Preference_Level'] = pd.cut(X['digital_savviness_score'],
                                                 bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
                                                 labels=['Traditional', 'Low_Digital',
                                                        'Moderate_Digital', 'Digital_Adopter',
                                                        'Highly_Digital'])

        # 3. Loan and Payment Profile
        if 'EMI_to_Income_Ratio' in X.columns:
            X['EMI_Burden_Level'] = pd.cut(X['EMI_to_Income_Ratio'],
                                          bins=[0, 0.2, 0.4, 0.6, float('inf')],
                                          labels=['Low', 'Moderate', 'High', 'Very_High'])

        if 'Credit_Utilization_Proxy' in X.columns:
            X['Credit_Utilization_Level'] = pd.cut(X['Credit_Utilization_Proxy'],
                                                 bins=[0, 0.3, 0.6, 0.9, 1.1],
                                                 labels=['Low', 'Moderate', 'High', 'Very_High'])

        # 4. Spending Behavior Profile
        if 'spend_to_income_ratio' in X.columns:
            X['Spending_Profile'] = pd.cut(X['spend_to_income_ratio'],
                                         bins=[0, 0.5, 0.8, 1.2, float('inf')],
                                         labels=['Conservative', 'Moderate', 'High', 'Very_High'])

        if 'Essential_vs_NonEssential_Ratio' in X.columns:
            X['Spending_Priority'] = pd.cut(X['Essential_vs_NonEssential_Ratio'],
                                          bins=[0, 1, 3, 6, float('inf')],
                                          labels=['Luxury', 'Balanced', 'Essential', 'Highly_Essential'])

        # 5. Payment Behavior Profile
        if 'Payment_Frequency_Regular' in X.columns and 'Payment_Frequency_Irregular' in X.columns:
            X['Payment_Behavior'] = np.where(X['Payment_Frequency_Regular'] == 1, 'Regular',
                                           np.where(X['Payment_Frequency_Irregular'] == 1, 'Irregular', 'Unknown'))

        # 6. Product Type Influence
        product_columns = [col for col in X.columns if 'Product_Type_' in col]
        if product_columns:
            # Identify primary product type
            X['Primary_Product_Type'] = X[product_columns].idxmax(axis=1)
            X['Primary_Product_Type'] = X['Primary_Product_Type'].str.replace('Product_Type_', '')

        # 7. Preferred Payment Channel
        payment_channel_cols = [col for col in X.columns if 'Preferred_Payment_Channel_' in col]
        if payment_channel_cols:
            X['Primary_Payment_Channel'] = X[payment_channel_cols].idxmax(axis=1)
            X['Primary_Payment_Channel'] = X['Primary_Payment_Channel'].str.replace('Preferred_Payment_Channel_', '')

        # 8. Flight Risk Profile
        if 'Flight_Risk_Score' in X.columns:
            X['Flight_Risk_Level'] = pd.cut(X['Flight_Risk_Score'],
                                          bins=[0, 0.3, 0.6, 0.8, 1.0],
                                          labels=['Low', 'Medium', 'High', 'Very_High'])

        # 9. Seasonal and Weekend Spending
        if 'Seasonal_Spend_Variation' in X.columns:
            X['Seasonal_Spend_Behavior'] = pd.cut(X['Seasonal_Spend_Variation'],
                                                bins=[-float('inf'), -0.1, 0.1, float('inf')],
                                                labels=['Low_Variation', 'Stable', 'High_Variation'])

        if 'Weekend_Spend_Ratio' in X.columns:
            X['Weekend_Spend_Profile'] = pd.cut(X['Weekend_Spend_Ratio'],
                                              bins=[0, 0.2, 0.4, 0.6, 1.0],
                                              labels=['Weekday_Spender', 'Balanced', 'Weekend_Spender', 'Heavy_Weekend_Spender'])

        # 10. Communication Behavior
        if 'Agent_Stickiness' in X.columns:
            X['Agent_Loyalty_Level'] = pd.cut(X['Agent_Stickiness'],
                                            bins=[0, 0.2, 0.5, 0.8, 1.0],
                                            labels=['Low', 'Medium', 'High', 'Very_High'])

        if 'Contact_Fatigue_Index' in X.columns:
            X['Contact_Fatigue_Level'] = pd.cut(X['Contact_Fatigue_Index'],
                                              bins=[0, 2, 5, 10, float('inf')],
                                              labels=['Low', 'Medium', 'High', 'Very_High'])

        return X

class EnhancedChannelEffectivenessCalculator(BaseEstimator, TransformerMixin):
    """Calculate how effective each channel is for each customer with comprehensive financial factors"""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Define the channels we're analyzing
        channels = ['SMS', 'Email', 'Call', 'WhatsApp', 'IVR', 'Field_Agent']

        # Get available positive response columns
        available_positive_responses = [col for col in X.columns if 'Response_Outcome' in col]
        print(f"Available response outcomes: {available_positive_responses}")

        # For each channel, calculate an effectiveness score with comprehensive adjustments
        for channel in channels:
            # Start with response rate (most important factor)
            response_col = f'Channel_used_{channel}'

            if response_col in X.columns:
                # Initialize effectiveness score
                effectiveness_score = np.zeros(len(X))

                # Calculate effectiveness based on usage and positive outcomes
                channel_used_mask = X[response_col] == 1
                if channel_used_mask.any() and available_positive_responses:
                    # Check if any positive response occurred when this channel was used
                    positive_mask = X[available_positive_responses].any(axis=1)
                    successful_uses = (channel_used_mask & positive_mask)

                    # Effectiveness: 1 if successful, 0.5 if used but not successful, 0 if not used
                    effectiveness_score = np.where(
                        successful_uses, 1.0,
                        np.where(channel_used_mask, 0.5, 0.0)
                    )
                elif channel_used_mask.any():
                    # If no response outcomes available, use usage as proxy
                    effectiveness_score = np.where(channel_used_mask, 0.7, 0.0)

                # Apply comprehensive financial profile adjustments
                effectiveness_score = self._apply_comprehensive_adjustments(X, channel, effectiveness_score)

                X[f'{channel}_Effectiveness'] = effectiveness_score

                # Add bonus for historical usage (more data = more reliable)
                historical_col = f'Contact_History_{channel}'
                if historical_col in X.columns:
                    # Normalize historical contact frequency
                    if X[historical_col].max() > 0:
                        usage_bonus = (X[historical_col] / X[historical_col].max()) * 0.1
                        X[f'{channel}_Effectiveness'] = X[f'{channel}_Effectiveness'] + usage_bonus
                        # Cap at 1.0
                        X[f'{channel}_Effectiveness'] = np.clip(X[f'{channel}_Effectiveness'], 0, 1.0)
            else:
                # If no usage data, set to neutral score with comprehensive adjustments
                base_score = 0.3
                adjusted_score = self._apply_comprehensive_adjustments(X, channel, np.full(len(X), base_score))
                X[f'{channel}_Effectiveness'] = adjusted_score

        return X

    def _apply_comprehensive_adjustments(self, X, channel, base_scores):
        """Apply comprehensive financial profile adjustments to channel effectiveness scores"""
        adjusted_scores = base_scores.copy()

        # 1. Digital Behavior Adjustments
        if 'digital_savviness_score' in X.columns:
            digital_channels = ['SMS', 'Email', 'WhatsApp', 'IVR']
            if channel in digital_channels:
                # Higher digital savviness = higher preference for digital channels
                digital_bonus = X['digital_savviness_score'] * 0.2
                adjusted_scores = adjusted_scores + digital_bonus
            else:
                # Non-digital channels (Call, Field_Agent) get bonus for low digital savviness
                traditional_bonus = (1 - X['digital_savviness_score']) * 0.15
                adjusted_scores = adjusted_scores + traditional_bonus

        # 2. Risk and Stress Adjustments
        if 'financial_stress_score' in X.columns:
            if channel in ['Call', 'Field_Agent']:
                # High stress customers may avoid intrusive channels
                stress_penalty = (X['financial_stress_score'] / 100) * 0.3
                adjusted_scores = adjusted_scores - stress_penalty
            elif channel in ['SMS', 'Email']:
                # Low-intrusion channels preferred by stressed customers
                stress_bonus = (X['financial_stress_score'] / 100) * 0.15
                adjusted_scores = adjusted_scores + stress_bonus

        # 3. AAR Risk Level Adjustments
        if 'aar_score' in X.columns:
            if channel in ['Call', 'Field_Agent']:
                # Higher risk customers may need more personal contact
                risk_bonus = X['aar_score'] * 0.15
                adjusted_scores = adjusted_scores + risk_bonus

        # 4. Loan and Payment Behavior Adjustments
        if 'EMI_to_Income_Ratio' in X.columns:
            high_emi_burden = X['EMI_to_Income_Ratio'] > 0.4
            if channel in ['Call', 'Field_Agent']:
                # High EMI burden customers may avoid personal contact
                emi_penalty = np.where(high_emi_burden, 0.15, 0)
                adjusted_scores = adjusted_scores - emi_penalty

        if 'Partial_Payment_Indicator' in X.columns:
            if channel in ['Call', 'Field_Agent']:
                # Customers with partial payments may need more follow-up
                partial_payment_bonus = X['Partial_Payment_Indicator'] * 0.1
                adjusted_scores = adjusted_scores + partial_payment_bonus

        # 5. Credit Utilization Adjustments
        if 'Credit_Utilization_Proxy' in X.columns:
            high_utilization = X['Credit_Utilization_Proxy'] > 0.7
            if channel in ['Call', 'Field_Agent']:
                # High credit utilization may indicate need for financial advice
                utilization_bonus = np.where(high_utilization, 0.1, 0)
                adjusted_scores = adjusted_scores + utilization_bonus

        # 6. Payment Behavior Adjustments
        if 'Payment_Behavior' in X.columns:
            if channel in ['SMS', 'Email']:
                # Regular payers may prefer automated reminders
                regular_bonus = np.where(X['Payment_Behavior'] == 'Regular', 0.1, 0)
                adjusted_scores = adjusted_scores + regular_bonus
            elif channel in ['Call', 'Field_Agent']:
                # Irregular payers may need personal follow-up
                irregular_bonus = np.where(X['Payment_Behavior'] == 'Irregular', 0.15, 0)
                adjusted_scores = adjusted_scores + irregular_bonus

        # 7. Product Type Adjustments
        if 'Primary_Product_Type' in X.columns:
            complex_products = ['Business loan', 'Auto loan', 'Education loan']
            if channel in ['Call', 'Field_Agent']:
                # Complex products may require more personal explanation
                complex_product_bonus = np.where(X['Primary_Product_Type'].isin(complex_products), 0.1, 0)
                adjusted_scores = adjusted_scores + complex_product_bonus

        # 8. Payment Channel Preferences
        if 'Primary_Payment_Channel' in X.columns:
            digital_payment_channels = ['UPI', 'Credit Card', 'Debit Card']
            if channel in ['SMS', 'Email', 'WhatsApp']:
                # Digital payment users may prefer digital communication
                digital_payment_bonus = np.where(X['Primary_Payment_Channel'].isin(digital_payment_channels), 0.1, 0)
                adjusted_scores = adjusted_scores + digital_payment_bonus
            elif channel == 'Call':
                # Cash users may prefer voice communication
                cash_bonus = np.where(X['Primary_Payment_Channel'] == 'Cash', 0.1, 0)
                adjusted_scores = adjusted_scores + cash_bonus

        # 9. Flight Risk Adjustments
        if 'Flight_Risk_Score' in X.columns:
            if channel in ['Call', 'Field_Agent']:
                # High flight risk customers may need personal retention efforts
                flight_risk_bonus = X['Flight_Risk_Score'] * 0.2
                adjusted_scores = adjusted_scores + flight_risk_bonus

        # 10. Spending Pattern Adjustments
        if 'Spending_Priority' in X.columns:
            if channel in ['Email', 'SMS']:
                # Essential spenders may prefer formal communication
                essential_bonus = np.where(X['Spending_Priority'].isin(['Essential', 'Highly_Essential']), 0.1, 0)
                adjusted_scores = adjusted_scores + essential_bonus

        if 'Weekend_Spend_Profile' in X.columns:
            if channel in ['SMS', 'WhatsApp']:
                # Weekend spenders may be more responsive to casual channels
                weekend_bonus = np.where(X['Weekend_Spend_Profile'].isin(['Weekend_Spender', 'Heavy_Weekend_Spender']), 0.1, 0)
                adjusted_scores = adjusted_scores + weekend_bonus

        # 11. Communication Behavior Adjustments
        if 'Agent_Loyalty_Level' in X.columns:
            if channel in ['Call', 'Field_Agent']:
                # High agent loyalty customers prefer personal contact
                loyalty_map = {'Low': 0, 'Medium': 0.05, 'High': 0.1, 'Very_High': 0.15}
                loyalty_bonus = X['Agent_Loyalty_Level'].map(loyalty_map).fillna(0).astype(float)
                adjusted_scores = adjusted_scores + loyalty_bonus

        if 'Contact_Fatigue_Level' in X.columns:
            if channel in ['Call', 'Field_Agent']:
                # High contact fatigue customers avoid intrusive channels
                fatigue_map = {'Low': 0, 'Medium': -0.05, 'High': -0.1, 'Very_High': -0.15}
                fatigue_penalty = X['Contact_Fatigue_Level'].map(fatigue_map).fillna(0).astype(float)
                adjusted_scores = adjusted_scores + fatigue_penalty
            elif channel in ['SMS', 'Email']:
                # Low-intrusion channels for fatigued customers
                fatigue_map = {'Low': 0, 'Medium': 0.05, 'High': 0.1, 'Very_High': 0.15}
                fatigue_bonus = X['Contact_Fatigue_Level'].map(fatigue_map).fillna(0).astype(float)
                adjusted_scores = adjusted_scores + fatigue_bonus

        # 12. Seasonal Spending Adjustments
        if 'Seasonal_Spend_Variation' in X.columns:
            if channel in ['SMS', 'WhatsApp']:
                # High variation spenders may respond to timely, quick communications
                variation_bonus = np.abs(X['Seasonal_Spend_Variation']) * 0.1
                adjusted_scores = adjusted_scores + variation_bonus

        # 13. Festive Spending Adjustments
        if 'Festive_Season_Spend_SGD' in X.columns:
            if channel in ['SMS', 'WhatsApp']:
                # High festive spenders may be more responsive to promotional channels
                festive_bonus = np.where(X['Festive_Season_Spend_SGD'] > X['Festive_Season_Spend_SGD'].median(), 0.1, 0)
                adjusted_scores = adjusted_scores + festive_bonus

        return np.clip(adjusted_scores, 0, 1.0)

class ChannelRanker(BaseEstimator, TransformerMixin):
    """Rank channels from best to worst for each customer"""

    def __init__(self):
        self.channels = ['SMS', 'Email', 'Call', 'WhatsApp', 'IVR', 'Field_Agent']

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Get effectiveness scores for all channels
        effectiveness_cols = [f'{channel}_Effectiveness' for channel in self.channels]

        # For each customer, rank channels by effectiveness
        preference_orders = []

        for i in range(len(X)):
            customer_scores = {}

            # Get scores for this customer
            for channel in self.channels:
                score_col = f'{channel}_Effectiveness'
                if score_col in X.columns:
                    customer_scores[channel] = X[score_col].iloc[i]
                else:
                    customer_scores[channel] = 0  # Default if missing

            # Sort channels by score (highest to lowest)
            ranked_channels = sorted(customer_scores.items(),
                                   key=lambda x: x[1],
                                   reverse=True)

            # Create preference order string
            preference_order = ','.join([channel for channel, score in ranked_channels])
            preference_orders.append(preference_order)

        # Add preference order to dataframe
        X['Channel_Preference_Order'] = preference_orders

        return X

class PreferenceLabelEncoder(BaseEstimator, TransformerMixin):
    """Convert preference orders into model-friendly formats"""

    def __init__(self):
        self.label_encoder = LabelEncoder()
        self.unique_orders = None

    def fit(self, X, y=None):
        if 'Channel_Preference_Order' in X.columns:
            # Learn all possible preference patterns
            self.unique_orders = X['Channel_Preference_Order'].unique()
            self.label_encoder.fit(self.unique_orders)
        return self

    def transform(self, X):
        X = X.copy()

        if 'Channel_Preference_Order' not in X.columns:
            print("No preference orders found. Run ChannelRanker first.")
            return X

        # Method 1: Encode entire order as one label (like animal classification)
        X['Preference_Label'] = self.label_encoder.transform(X['Channel_Preference_Order'])

        # Method 2: Create simple "top choice" label
        X['Top_Channel'] = X['Channel_Preference_Order'].str.split(',').str[0]

        # Method 3: Binary indicators for each channel being in top 3
        for channel in ['SMS', 'Email', 'Call', 'WhatsApp', 'IVR', 'Field_Agent']:
            X[f'Prefers_{channel}_Top3'] = X['Channel_Preference_Order'].apply(
                lambda x: 1 if channel in x.split(',')[:3] else 0
            )

        return X

class ComprehensiveLabelConstructionPipeline:
    """Complete pipeline for creating channel preference labels with comprehensive financial factors"""

    def __init__(self):
        self.pipeline = None
        self.labels = None
        self._build_pipeline()

    def _build_pipeline(self):
        """Build the label construction pipeline"""

        self.pipeline = Pipeline([
            # Step 0: Calculate comprehensive financial profile metrics
            ('financial_profile', ComprehensiveFinancialProfileCalculator()),

            # Step 1: Calculate how good each channel is (with comprehensive adjustments)
            ('effectiveness_calculator', EnhancedChannelEffectivenessCalculator()),

            # Step 2: Rank channels from best to worst
            ('channel_ranker', ChannelRanker()),

            # Step 3: Convert to model-friendly formats
            ('label_encoder', PreferenceLabelEncoder())
        ])

    def fit_transform(self, X, y=None):
        """Create labels from the engineered features"""
        print("=" * 60)
        print("BUILDING COMPREHENSIVE CHANNEL PREFERENCE LABELS")
        print("=" * 60)

        try:
            # Store Customer_id for merging later
            if 'Customer_id' in X.columns:
                customer_ids = X['Customer_id'].copy()
            else:
                # Create temporary customer IDs if not present
                customer_ids = pd.Series([f'CUST_{i+1:06d}' for i in range(len(X))])
                X['Customer_id'] = customer_ids

            # Apply the pipeline
            X_with_labels = self.pipeline.fit_transform(X)

            # Extract the important label columns
            initial_label_cols = ['Customer_id', 'Channel_Preference_Order',
                                  'Preference_Label', 'Top_Channel']

            # Collect all potential label-related columns
            all_potential_label_cols = []
            all_potential_label_cols.extend(initial_label_cols)
            all_potential_label_cols.extend([col for col in X_with_labels.columns if 'Prefers_' in col])
            all_potential_label_cols.extend([col for col in X_with_labels.columns if 'Effectiveness' in col])
            all_potential_label_cols.extend([col for col in X_with_labels.columns if
                                            any(term in col for term in ['AAR_', 'Financial_', 'Digital_', 'EMI_', 'Credit_',
                                            'Spending_', 'Payment_', 'Primary_', 'Flight_',
                                            'Seasonal_', 'Weekend_', 'Agent_', 'Contact_'])])

            # Ensure unique column names in the final selection
            final_label_columns_unique = []
            seen_cols = set()
            for col in all_potential_label_cols:
                if col not in seen_cols and col in X_with_labels.columns:
                    final_label_columns_unique.append(col)
                    seen_cols.add(col)

            self.labels = X_with_labels[final_label_columns_unique]

            print("✅ Comprehensive label construction completed!")
            print(f"Number of customers: {len(self.labels)}")
            print(f"Number of unique preference patterns: {len(self.labels['Channel_Preference_Order'].unique())}")

            # Show what we created
            self._analyze_labels()

            return X_with_labels, self.labels

        except Exception as e:
            print(f"❌ Error in label construction: {e}")
            import traceback
            traceback.print_exc()
            return X, None

    def _analyze_labels(self):
        """Analyze the created labels with comprehensive financial insights"""
        print("\n" + "="*50)
        print("COMPREHENSIVE LABEL ANALYSIS")
        print("="*50)

        # Top channels overall
        top_channels = self.labels['Top_Channel'].value_counts()
        print("\nMost popular top channels:")
        for channel, count in top_channels.items():
            percentage = (count / len(self.labels)) * 100
            print(f"   {channel}: {count} customers ({percentage:.1f}%) ")

        # Preference order distribution
        print(f"\nUnique preference orders: {len(self.labels['Channel_Preference_Order'].unique())}")

        # Show most common preference patterns
        common_orders = self.labels['Channel_Preference_Order'].value_counts().head(5)
        print("\n🔝 Top 5 preference patterns:")
        for order, count in common_orders.items():
            print(f"   {order}: {count} customers")

        # Effectiveness score statistics
        # Ensure we only consider unique effectiveness column names for analysis
        unique_effectiveness_cols = list(set([col for col in self.labels.columns if 'Effectiveness' in col]))
        if unique_effectiveness_cols:
            print("")
            for col in unique_effectiveness_cols: # Iterate over unique names
                channel = col.replace('_Effectiveness', '')
                avg_score = self.labels[col].mean()
                try:
                    print(f"")
                except (TypeError, ValueError):
                    print(f"")

        # Comprehensive financial profile vs channel preferences
        self._analyze_comprehensive_correlations()

    def _analyze_comprehensive_correlations(self):
        """Analyze how comprehensive financial factors correlate with channel preferences"""

        # Digital savviness vs digital channel preference
        if 'digital_savviness_score' in self.labels.columns:
            digital_channels = ['SMS', 'Email', 'WhatsApp', 'IVR']
            digital_pref_scores = []

            for channel in digital_channels:
                if f'Prefers_{channel}_Top3' in self.labels.columns:
                    mask = self.labels[f'Prefers_{channel}_Top3'] == 1
                    if mask.any():
                        avg_digital_score = self.labels.loc[mask, 'digital_savviness_score'].mean()
                        digital_pref_scores.append((channel, avg_digital_score))

            if digital_pref_scores:
                for channel, score in sorted(digital_pref_scores, key=lambda x: x[1], reverse=True):
                    print(f"")

        # Financial stress vs channel preference
        if 'financial_stress_score' in self.labels.columns:
            channels = ['Call', 'Field_Agent', 'SMS', 'Email']
            stress_pref_scores = []

            for channel in channels:
                if f'Prefers_{channel}_Top3' in self.labels.columns:
                    mask = self.labels[f'Prefers_{channel}_Top3'] == 1
                    if mask.any():
                        avg_stress_score = self.labels.loc[mask, 'financial_stress_score'].mean()
                        stress_pref_scores.append((channel, avg_stress_score))

            if stress_pref_scores:
                for channel, score in sorted(stress_pref_scores, key=lambda x: x[1], reverse=True):
                    print(f"")

        # Payment behavior vs channel preference
        if 'Payment_Behavior' in self.labels.columns:
            payment_behavior_analysis = self.labels.groupby(['Payment_Behavior', 'Top_Channel']).size().unstack(fill_value=0)


        # Product type vs channel preference
        if 'Primary_Product_Type' in self.labels.columns:
            product_channel_analysis = self.labels.groupby(['Primary_Product_Type', 'Top_Channel']).size().reset_index(name='Count')
            product_channel_analysis = product_channel_analysis.sort_values('Count', ascending=False).head(5)
            for _, row in product_channel_analysis.iterrows():
                print(f"")

    def get_labels(self):
        """Get the constructed labels"""
        return self

# 🚀 **MAIN EXECUTION**

if __name__ == "__main__":
    # Load your engineered features
    try:
        df = pd.read_csv('engineered_features.csv')
        print("✅ Loaded engineered_features.csv")
    except FileNotFoundError:
        try:
            df = pd.read_csv('feature_engineered.csv')
            print("✅ Loaded feature_engineered.csv")
        except FileNotFoundError:
            print("❌ Could not find engineered features file")
            exit()

    print("Loaded engineered features:")
    print(f"   Shape: {df.shape}")
    print(f"   Columns: {len(df.columns)}")
    print(f"   Customers: {len(df)}")

    # Check for comprehensive financial columns
    comprehensive_financial_columns = [
        'aar_score', 'customer_risk_level', 'finance_stress_status', 'financial_stress_score',
        'financial_health_status', 'financial_health_score', 'spend_to_income_ratio',
        'digital_savviness_score', 'digital_savviness_level', 'digital_transaction_ratio',
        'recurring_transaction_ratio', 'app_usage_factor', 'online_banking_factor',
        'channel_preference_score', 'EMI_to_Income_Ratio', 'Credit_Utilization_Proxy',
        'Income_Stability_Score', 'Essential_vs_NonEssential_Ratio', 'Agent_Stickiness',
        'Contact_Fatigue_Index', 'Area_Income_Percentile', 'Amount_Paid_Each_Month_SGD',
        'Festive_Season_Spend_SGD', 'Flight_Risk_Score', 'Partial_Payment_Indicator',
        'Payment_Frequency_Irregular', 'Payment_Frequency_Regular',
        'Preferred_Payment_Channel_Cash', 'Preferred_Payment_Channel_Credit Card',
        'Preferred_Payment_Channel_Debit Card', 'Preferred_Payment_Channel_UPI',
        'Product_Type_Auto loan', 'Product_Type_Business loan', 'Product_Type_Credit card',
        'Product_Type_Education loan', 'Product_Type_Personal loan', 'Seasonal_Spend_Variation',
        'Weekend_Spend_Ratio'
    ]

    available_financial_cols = [col for col in comprehensive_financial_columns if col in df.columns]
    print(f"\nAvailable comprehensive financial columns: {len(available_financial_cols)}/{len(comprehensive_financial_columns)}")
    print(f"Available: {available_financial_cols}")

    # Check for required channel columns
    required_channel_cols = ['Channel_used_SMS', 'Channel_used_Email', 'Channel_used_Call',
                           'Channel_used_WhatsApp', 'Channel_used_IVR', 'Channel_used_Field Agent']

    available_channel_cols = [col for col in required_channel_cols if col in df.columns]
    print(f"\nAvailable channel usage columns: {len(available_channel_cols)}/{len(required_channel_cols)}")

    # Check for response outcome columns
    response_cols = [col for col in df.columns if 'Response_Outcome' in col]
    print(f"Available response outcome columns: {response_cols}")

    # Check for contact history columns
    contact_history_cols = [col for col in df.columns if 'Contact_History' in col]
    print(f"Available contact history columns: {contact_history_cols}")

    # Use the comprehensive pipeline with all financial factors
    label_pipeline = ComprehensiveLabelConstructionPipeline()
    df_with_labels, labels = label_pipeline.fit_transform(df)

    if labels is not None:
        # Save results
        labels.to_csv('channel_preference_labels.csv', index=False)
        df_with_labels.to_csv('features_with_labels.csv', index=False)

        print("\n✅ Saved results:")
        print("   - channel_preference_labels.csv (labels + comprehensive factors)")
        print("   - features_with_labels.csv (all features + labels)")

        # Show sample of what we created
        print("\n📊 Sample of comprehensive labels:")
        sample_cols = ['Customer_id', 'Top_Channel', 'Channel_Preference_Order']
        financial_sample_cols = ['financial_stress_score', 'digital_savviness_score', 'EMI_to_Income_Ratio', 'Flight_Risk_Score']

        available_sample_cols = [col for col in sample_cols if col in labels.columns]
        available_financial_cols = [col for col in financial_sample_cols if col in labels.columns]

        display_cols = available_sample_cols + available_financial_cols
        print(labels[display_cols].head(10))

        # Show effectiveness scores with comprehensive adjustments
        effectiveness_cols = [col for col in labels.columns if 'Effectiveness' in col]
        if effectiveness_cols:
            print("\n🔢 Sample effectiveness scores (with comprehensive adjustments):")
            effectiveness_sample = labels[['Customer_id'] + effectiveness_cols].head(5)
            print(effectiveness_sample)

    else:
        print("❌ Label creation failed!")

✅ Loaded engineered_features.csv
Loaded engineered features:
   Shape: (100000, 111)
   Columns: 111
   Customers: 100000

Available comprehensive financial columns: 38/38
Available: ['aar_score', 'customer_risk_level', 'finance_stress_status', 'financial_stress_score', 'financial_health_status', 'financial_health_score', 'spend_to_income_ratio', 'digital_savviness_score', 'digital_savviness_level', 'digital_transaction_ratio', 'recurring_transaction_ratio', 'app_usage_factor', 'online_banking_factor', 'channel_preference_score', 'EMI_to_Income_Ratio', 'Credit_Utilization_Proxy', 'Income_Stability_Score', 'Essential_vs_NonEssential_Ratio', 'Agent_Stickiness', 'Contact_Fatigue_Index', 'Area_Income_Percentile', 'Amount_Paid_Each_Month_SGD', 'Festive_Season_Spend_SGD', 'Flight_Risk_Score', 'Partial_Payment_Indicator', 'Payment_Frequency_Irregular', 'Payment_Frequency_Regular', 'Preferred_Payment_Channel_Cash', 'Preferred_Payment_Channel_Credit Card', 'Preferred_Payment_Channel_Debit Card'

In [7]:
import pandas as pd

try:
    # Load the features_with_labels DataFrame
    file_path = 'features_with_labels.csv'
    df_features_with_labels = pd.read_csv(file_path)
    print(f"Loaded {len(df_features_with_labels)} rows from {file_path}")

    # List of columns to drop as requested by the user
    columns_to_drop = [
        'Channel_used_Call', 'Channel_used_Email', 'Channel_used_Field Agent',
        'Channel_used_IVR', 'Channel_used_SMS', 'Channel_used_WhatsApp',
        'Contact_History_Call_Attempts', 'Contact_History_EmailLogs',
        'Contact_History_FieldAgent', 'Contact_History_IVR', 'Contact_History_SMS',
        'Contact_History_WhatsApp', 'Response_Outcome_Connected', 'Response_Outcome_Disconnected',
        'Response_Outcome_Ignored', 'Response_Outcome_Partial paid',
        'Response_Outcome_Promised to pay', 'AAR_Risk_Level',
        'Financial_Stress_Level', 'Financial_Health_Level', 'Digital_Preference_Level',
        'EMI_Burden_Level', 'Credit_Utilization_Level', 'Spending_Profile',
        'Spending_Priority', 'Payment_Behavior', 'Primary_Product_Type',
        'Primary_Payment_Channel', 'Flight_Risk_Level', 'Seasonal_Spend_Behavior',
        'Weekend_Spend_Profile', 'Agent_Loyalty_Level', 'Contact_Fatigue_Level',
        'Prefers_SMS_Top3', 'Prefers_Email_Top3', 'Prefers_Call_Top3',
        'Prefers_WhatsApp_Top3', 'Prefers_IVR_Top3', 'Prefers_Field_Agent_Top3'
    ]

    # Filter to only drop columns that exist in the DataFrame
    existing_columns_to_drop = [col for col in columns_to_drop if col in df_features_with_labels.columns]

    if existing_columns_to_drop:
        print(f"Dropping {len(existing_columns_to_drop)} columns...")
        df_features_with_labels.drop(columns=existing_columns_to_drop, inplace=True)
        df_features_with_labels.to_csv(file_path, index=False)
        print(f"Updated data saved to '{file_path}'")
        print(f"New column count: {len(df_features_with_labels.columns)}")
        print("Sample of updated data:")
        display(df_features_with_labels.head())
    else:
        print("No specified columns found to drop or they were already removed.")

except FileNotFoundError:
    print(f"Error: '{file_path}' not found. Please ensure the file exists.")
except Exception as e:
    print(f"An error occurred: {e}")


Loaded 100000 rows from features_with_labels.csv
Dropping 39 columns...
Updated data saved to 'features_with_labels.csv'
New column count: 103
Sample of updated data:


,Customer_id,aar_score,customer_risk_level,finance_stress_status,financial_stress_score,financial_health_status,financial_health_score,spend_to_income_ratio,digital_savviness_score,digital_savviness_level,...,Weekend_Spend_Ratio,SMS_Effectiveness,Email_Effectiveness,Call_Effectiveness,WhatsApp_Effectiveness,IVR_Effectiveness,Field_Agent_Effectiveness,Channel_Preference_Order,Preference_Label,Top_Channel
0,SCB909522509,0.580218,Medium,Medium stress,32.00,Moderate,1,0.63,1.0000,Highly Digital,...,0.40,0.534526,0.498000,0.000000,1.000000,0.216667,0.243033,"WhatsApp,SMS,Email,Field_Agent,IVR,Call",69,WhatsApp
1,SCB982519764,0.558600,Medium,Medium stress,41.50,Moderate,2,0.54,0.5953,Moderate Digital,...,0.33,0.370363,0.331310,0.059995,0.260137,0.152393,0.359995,"SMS,Field_Agent,Email,WhatsApp,IVR,Call",55,SMS
2,SCB934804341,0.561273,Medium,Medium stress,40.50,Healthy,3,0.76,0.5411,Moderate Digital,...,0.34,0.459496,0.418970,1.000000,0.245912,0.124887,0.541526,"Call,Field_Agent,SMS,Email,WhatsApp,IVR",1,Call
3,SCB929520634,0.597188,Medium,Medium stress,36.50,Moderate,2,0.57,0.9708,Highly Digital,...,0.34,0.419173,0.398910,0.328458,0.324545,0.194160,0.628458,"Field_Agent,SMS,Email,Call,WhatsApp,IVR",24,Field_Agent
4,SCB930205962,0.463050,Low,High stress,63.05,Moderate,1,0.55,0.3879,Low Digital,...,0.40,0.305155,0.272155,1.000000,0.210580,0.077580,0.700123,"Call,Field_Agent,SMS,Email,WhatsApp,IVR",1,Call


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

class ChannelRankingDataPreparator:
    """Prepare data for XGBRanker training"""

    def __init__(self):
        self.channels = ['SMS', 'Email', 'Call', 'WhatsApp', 'IVR', 'Field_Agent']
        self.label_encoders = {}
        self.feature_cols = None # Store the feature columns

    def prepare_ranking_data(self, df):
        """Convert preference data to ranking format"""
        print("Preparing ranking format...")

        # Get feature columns (exclude label columns and Customer_id)
        exclude_cols = ['Customer_id', 'Channel_Preference_Order', 'Preference_Label', 'Top_Channel']
        exclude_cols.extend([col for col in df.columns if 'Prefers_' in col])

        self.feature_cols = [col for col in df.columns if col not in exclude_cols]

        # Encode categorical columns
        X = df[self.feature_cols].copy()
        categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

        if categorical_cols:
            print(f"Encoding {len(categorical_cols)} categorical columns...")
            for col in categorical_cols:
                le = LabelEncoder()
                X[col] = le.fit_transform(X[col].astype(str))
                self.label_encoders[col] = le

        # Create ranking dataset
        ranking_data = []
        group_sizes = []

        print(f"Processing {len(df)} customers...")

        for idx, (_, row) in enumerate(df.iterrows()):
            if (idx + 1) % 10000 == 0:
                print(f"Processed {idx + 1:,} customers")

            customer_id = row['Customer_id']
            preference_order = row['Channel_Preference_Order'].split(',')

            # Get customer features
            customer_features = X.iloc[idx].values

            # Create one sample per channel
            for rank, channel in enumerate(preference_order):
                if channel in self.channels:
                    # Channel one-hot encoding
                    channel_features = np.zeros(len(self.channels))
                    channel_idx = self.channels.index(channel)
                    channel_features[channel_idx] = 1

                    # Combine customer and channel features
                    combined_features = np.concatenate([customer_features, channel_features])

                    # Relevance score (higher rank = higher relevance)
                    relevance = len(self.channels) - rank

                    ranking_data.append({
                        'customer_id': customer_id,
                        'channel': channel,
                        'features': combined_features,
                        'relevance': relevance,
                        'group_id': idx
                    })

            group_sizes.append(len(self.channels))

        # Convert to arrays
        X_ranking = np.array([item['features'] for item in ranking_data])
        y_ranking = np.array([item['relevance'] for item in ranking_data])
        groups = np.array(group_sizes)

        print(f"\nRanking dataset created:")
        print(f"Total samples: {len(X_ranking):,}")
        print(f"Total customers (groups): {len(groups):,}")
        print(f"Features per sample: {X_ranking.shape[1]}")
        print(f"Group sizes (samples per customer): {groups[0]} (all should be {len(self.channels)})")

        return X_ranking, y_ranking, groups

def train_xgb_ranker(X, y, groups, test_size=0.2, random_state=42):
    """Train XGBRanker model"""
    print("\n" + "="*60)
    print("TRAINING XGBRANKER MODEL")
    print("="*60)

    # Calculate split points for groups
    n_train_groups = int(len(groups) * (1 - test_size))
    train_samples = sum(groups[:n_train_groups])

    # Split data
    X_train = X[:train_samples]
    y_train = y[:train_samples]
    groups_train = groups[:n_train_groups]

    X_test = X[train_samples:]
    y_test = y[train_samples:]
    groups_test = groups[n_train_groups:]

    print(f"Feature matrix shape: {X_train.shape}")
    print(f"Label vector shape: {y_train.shape}")
    print(f"Number of groups: {len(groups_train):,}")
    print(f"\nTrain set: {len(X_train):,} samples from {len(groups_train):,} customers")
    print(f"Test set: {len(X_test):,} samples from {len(groups_test):,} customers")

    # Initialize XGBRanker
    print("\nInitializing XGBRanker...")
    model = xgb.XGBRanker(
        objective='rank:ndcg',
        learning_rate=0.1,
        max_depth=6,
        n_estimators=100,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=random_state,
        eval_metric='ndcg@32'
    )

    # Train model
    print("Training XGBRanker...")
    model.fit(
        X_train, y_train,
        group=groups_train,
        eval_set=[(X_test, y_test)],
        eval_group=[groups_test],
        verbose=True
    )

    print("\nModel training completed successfully!")

    # Make predictions
    print("\nMaking predictions...")
    y_pred = model.predict(X_test)

    # Calculate NDCG
    print("Calculating NDCG score...")
    # Reshape predictions for NDCG calculation
    y_test_reshaped = []
    y_pred_reshaped = []

    start_idx = 0
    for group_size in groups_test:
        end_idx = start_idx + group_size
        y_test_reshaped.append(y_test[start_idx:end_idx])
        y_pred_reshaped.append(y_pred[start_idx:end_idx])
        start_idx = end_idx

    ndcg = ndcg_score(y_test_reshaped, y_pred_reshaped)
    print(f"NDCG Score: {ndcg:.4f}")

    return model, ndcg

def demonstrate_predictions(model, X, channels, n_customers=5):
    """Show example predictions"""
    print("\n" + "="*60)
    print("EXAMPLE PREDICTIONS")
    print("="*60)

    samples_per_customer = len(channels)

    for i in range(n_customers):
        start_idx = i * samples_per_customer
        end_idx = start_idx + samples_per_customer

        customer_samples = X[start_idx:end_idx]
        predictions = model.predict(customer_samples)

        # Sort channels by prediction score
        channel_scores = list(zip(channels, predictions))
        channel_scores.sort(key=lambda x: x[1], reverse=True)

        print(f"\nCustomer {i+1} Channel Preferences:")
        for rank, (channel, score) in enumerate(channel_scores, 1):
            print(f"  {rank}. {channel}: {score:.4f}")

# Main execution
if __name__ == "__main__":
    print("="*60)
    print("PREPARING CHANNEL RANKING DATA FOR XGBRANKER")
    print("="*60)

    # Load data
    print("Loading features_with_labels.csv...")
    try:
        df = pd.read_csv('features_with_labels.csv')
        print(f"Loaded data shape: {df.shape}")
    except FileNotFoundError:
        print("Error: features_with_labels.csv not found!")
        exit(1)

    # Prepare data
    preparator = ChannelRankingDataPreparator()
    X, y, groups = preparator.prepare_ranking_data(df)

    # Train model
    model, ndcg_score = train_xgb_ranker(X, y, groups)

    # Save model and preparator
    model_filename = 'xgb_channel_ranker.pkl'
    joblib.dump(model, model_filename)
    joblib.dump(preparator, 'channel_ranking_data_preparator.pkl') # Save the preparator
    print(f"\nModel saved as '{model_filename}'")
    print("Preparator saved as 'channel_ranking_data_preparator.pkl'")


    # Demonstrate predictions
    demonstrate_predictions(model, X, preparator.channels)

    print("\n" + "="*60)
    print("MODEL TRAINING COMPLETED SUCCESSFULLY!")
    print("="*60)
    print(f"Feature columns used: {X.shape[1]}")
    print(f"Model saved as: {model_filename}")
    print(f"Total customers processed: {len(df):,}")
    print(f"Final NDCG Score: {ndcg_score:.4f}")

PREPARING CHANNEL RANKING DATA FOR XGBRANKER
Loading features_with_labels.csv...
Loaded data shape: (100000, 103)
Preparing ranking format...
Encoding 9 categorical columns...
Processing 100000 customers...
Processed 10,000 customers
Processed 20,000 customers
Processed 30,000 customers
Processed 40,000 customers
Processed 50,000 customers
Processed 60,000 customers
Processed 70,000 customers
Processed 80,000 customers
Processed 90,000 customers
Processed 100,000 customers

Ranking dataset created:
Total samples: 600,000
Total customers (groups): 100,000
Features per sample: 105
Group sizes (samples per customer): 6 (all should be 6)

TRAINING XGBRANKER MODEL
Feature matrix shape: (480000, 105)
Label vector shape: (480000,)
Number of groups: 80,000

Train set: 480,000 samples from 80,000 customers
Test set: 120,000 samples from 20,000 customers

Initializing XGBRanker...
Training XGBRanker...
[0]	validation_0-ndcg@32:0.93052
[1]	validation_0-ndcg@32:0.96134
[2]	validation_0-ndcg@32:0.9

In [9]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import ndcg_score
from sklearn.preprocessing import LabelEncoder
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

class LightGBMRankerTrainer:
    """Train LightGBM model for channel preference ranking"""

    def __init__(self):
        self.model = None
        self.channels = ['SMS', 'Email', 'Call', 'WhatsApp', 'IVR', 'Field_Agent']
        self.label_encoders = {}

    def load_and_prepare_data(self, data_path):
        """Load and prepare data for LightGBM ranking"""
        print(" Loading data...")
        df = pd.read_csv(data_path)
        print(f" Loaded {len(df):,} customers with {len(df.columns):,} features")

        # Get feature columns (exclude label columns)
        exclude_cols = ['Customer_id', 'Channel_Preference_Order', 'Preference_Label', 'Top_Channel']
        exclude_cols.extend([col for col in df.columns if 'Prefers_' in col])
        feature_cols = [col for col in df.columns if col not in exclude_cols]

        # Store feature columns for prediction
        self.feature_cols = feature_cols

        # Encode categorical features
        X = df[feature_cols].copy()
        categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

        print(f"🔧 Encoding {len(categorical_cols)} categorical columns...")
        for col in categorical_cols:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            self.label_encoders[col] = le

        # Create ranking dataset
        print("Converting to ranking format...")
        ranking_data = []
        group_sizes = []

        for idx, (_, row) in enumerate(df.iterrows()):
            if (idx + 1) % 1000 == 0:
                print(f"   Processed {idx + 1:,} customers...")

            customer_id = row['Customer_id']
            preference_order = row['Channel_Preference_Order'].split(',')

            customer_features = X.iloc[idx].values

            for rank, channel in enumerate(preference_order):
                if channel in self.channels:
                    # Channel one-hot encoding
                    channel_features = np.zeros(len(self.channels))
                    channel_idx = self.channels.index(channel)
                    channel_features[channel_idx] = 1

                    combined_features = np.concatenate([customer_features, channel_features])
                    relevance = len(self.channels) - rank  # Higher rank = higher relevance

                    ranking_data.append({
                        'customer_id': customer_id,
                        'channel': channel,
                        'features': combined_features,
                        'relevance': relevance,
                        'group_id': idx
                    })

            group_sizes.append(len(self.channels))

        X_ranking = np.array([item['features'] for item in ranking_data])
        y_ranking = np.array([item['relevance'] for item in ranking_data])
        groups = np.array(group_sizes)

        print(f"Ranking dataset: {len(X_ranking):,} samples, {len(groups):,} customer groups")
        return X_ranking, y_ranking, groups, df

    def train_model(self, X, y, groups, test_size=0.2):
        """Train LightGBM ranking model"""
        print("\n Training LightGBM Ranker...")

        # Split data by groups (customers)
        n_train_groups = int(len(groups) * (1 - test_size))
        train_samples = sum(groups[:n_train_groups])

        X_train, X_test = X[:train_samples], X[train_samples:]
        y_train, y_test = y[:train_samples], y[train_samples:]
        groups_train, groups_test = groups[:n_train_groups], groups[n_train_groups:]

        print(f" Train set: {len(X_train):,} samples from {len(groups_train):,} customers")
        print(f" Test set:  {len(X_test):,} samples from {len(groups_test):,} customers")

        # Create LightGBM datasets
        train_data = lgb.Dataset(X_train, label=y_train, group=groups_train)
        test_data = lgb.Dataset(X_test, label=y_test, group=groups_test, reference=train_data)

        # LightGBM parameters for LambdaRank
        params = {
            'objective': 'lambdarank',
            'metric': 'ndcg',
            'ndcg_eval_at': [3, 5, 10],
            'learning_rate': 0.1,
            'num_leaves': 31,
            'max_depth': 8,
            'min_data_in_leaf': 20,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'lambda_l1': 0.1,
            'lambda_l2': 0.1,
            'verbosity': -1,
            'random_state': 42
        }

        # Train model
        print(" Training in progress...")
        self.model = lgb.train(
            params,
            train_data,
            num_boost_round=200,
            valid_sets=[test_data],
            valid_names=['test'],
            callbacks=[
                lgb.early_stopping(20),
                lgb.log_evaluation(50)  # Print every 50 rounds
            ]
        )

        return X_test, y_test, groups_test

    def evaluate_model(self, X_test, y_test, groups_test):
        """Evaluate the trained model"""
        print("\n Evaluating model performance...")

        # Make predictions
        y_pred = self.model.predict(X_test)

        # Calculate NDCG
        ndcg = self.calculate_ndcg(y_test, y_pred, groups_test)
        print(f"✅ NDCG Score: {ndcg:.4f}")

        # Feature importance
        # Note: Feature names from LightGBM are 0-based indices by default.
        # To get meaningful names, we would need to map them back to feature_cols.
        feature_importance = self.model.feature_importance(importance_type='gain')
        print(f" Number of features: {len(feature_importance)}")

        return ndcg, y_pred

    def calculate_ndcg(self, y_true, y_pred, groups):
        """Calculate NDCG score for ranking evaluation"""
        y_true_reshaped = []
        y_pred_reshaped = []

        start_idx = 0
        for group_size in groups:
            end_idx = start_idx + group_size
            y_true_reshaped.append(y_true[start_idx:end_idx])
            y_pred_reshaped.append(y_pred[start_idx:end_idx])
            start_idx = end_idx

        return ndcg_score(y_true_reshaped, y_pred_reshaped)

    def plot_training_metrics(self):
        """Plot training history and metrics"""
        if not hasattr(self.model, 'best_score'):
            print("  No training history available for plotting")
            return

        # Plot feature importance
        plt.figure(figsize=(12, 8))

        # Feature importance
        importance = self.model.feature_importance(importance_type='gain')
        # Get actual feature names from the stored feature_cols, excluding the channel one-hots
        base_feature_names = self.feature_cols
        # LightGBM feature importance includes the channel one-hots, need to adjust names
        all_feature_names = base_feature_names + [f'Channel_{ch}_onehot' for ch in self.channels]


        importance_df = pd.DataFrame({
            'feature': all_feature_names,
            'importance': importance
        }).sort_values('importance', ascending=False).head(20)

        plt.subplot(2, 1, 1) # Adjusted subplot layout
        sns.barplot(data=importance_df, x='importance', y='feature')
        plt.title('Top 20 Feature Importance (Gain)')
        plt.xlabel('Importance (Gain)')
        plt.ylabel('Feature')


        # Training history (if available)
        try:
            eval_results = self.model.evals_result_
            if 'test' in eval_results and 'ndcg@3' in eval_results['test']:
                plt.subplot(2, 1, 2) # Adjusted subplot layout
                plt.plot(eval_results['test']['ndcg@3'], label='NDCG@3')
                if 'ndcg@5' in eval_results['test']:
                     plt.plot(eval_results['test']['ndcg@5'], label='NDCG@5')
                if 'ndcg@10' in eval_results['test']:
                    plt.plot(eval_results['test']['ndcg@10'], label='NDCG@10')

                plt.title('Validation NDCG during Training')
                plt.xlabel('Iteration')
                plt.ylabel('NDCG')
                plt.legend()
                plt.grid(True) # Add grid for better readability
        except:
            print(" Could not plot training history.")
            pass

        plt.tight_layout()
        plt.savefig('lightgbm_training_metrics.png', dpi=300, bbox_inches='tight')
        plt.show()

    def save_model(self, model_path='lightgbm_channel_ranker.pkl'):
        """Save the trained model and preprocessing objects"""
        model_artifacts = {
            'model': self.model,
            'label_encoders': self.label_encoders,
            'channels': self.channels,
            'feature_cols': self.feature_cols # Save feature columns
        }

        joblib.dump(model_artifacts, model_path)
        print(f" Model saved as '{model_path}'")

    def predict_single_customer(self, customer_data):
        """Predict channel preferences for a single customer"""
        if self.model is None:
            print(" Model not trained yet!")
            return None

        # customer_data is expected to be a DataFrame containing one row for the customer
        if len(customer_data) != 1:
             print(" Expected customer_data to contain exactly one row.")
             return None

        # Select relevant feature columns
        X_customer_df = customer_data[self.feature_cols].copy()

        # Encode categorical features for this single row
        categorical_cols = X_customer_df.select_dtypes(include=['object']).columns.tolist()

        for col in categorical_cols:
            if col in self.label_encoders:
                le = self.label_encoders[col]
                # Transform the single value in the DataFrame column
                try:
                     X_customer_df[col] = le.transform(X_customer_df[col].astype(str))
                except ValueError as e:
                     print(f"Warning: Could not encode categorical value for column '{col}'. Error: {e}")
                     # Handle unseen categories - might need a strategy here (e.g., use a default, or drop)
                     # For now, let's replace with a placeholder or NaN
                     X_customer_df[col] = -1 # Use -1 for unseen categories

        # Extract customer features as a numpy array from the single row
        customer_features = X_customer_df.iloc[0].values

        # Create samples for each channel
        ranking_data = []
        for channel in self.channels:
            channel_features = np.zeros(len(self.channels))
            channel_idx = self.channels.index(channel)
            channel_features[channel_idx] = 1

            combined_features = np.concatenate([customer_features, channel_features])
            ranking_data.append({
                'channel': channel,
                'features': combined_features
            })

        X_ranking = np.array([item['features'] for item in ranking_data])

        # Make predictions
        predictions = self.model.predict(X_ranking)

        # Combine results
        channel_predictions = []
        for i, data in enumerate(ranking_data):
            channel_predictions.append({
                'channel': data['channel'],
                'score': predictions[i]
            })

        # Sort by score
        channel_predictions.sort(key=lambda x: x['score'], reverse=True)
        return channel_predictions

def main():
    """Main training function"""
    print("="*70)
    print(" LIGHTGBM CHANNEL PREFERENCE RANKER TRAINING")
    print("="*70)

    # Initialize trainer
    trainer = LightGBMRankerTrainer()

    # Load and prepare data
    data_path = 'features_with_labels.csv'  # Your engineered features file
    X, y, groups, df = trainer.load_and_prepare_data(data_path)

    # Train model
    X_test, y_test, groups_test = trainer.train_model(X, y, groups)

    # Evaluate model
    ndcg, predictions = trainer.evaluate_model(X_test, y_test, groups_test)

    # Save model
    trainer.save_model()

    # Demo prediction
    print("\n DEMO PREDICTION:")
    # Ensure the customer_id exists in the dataframe loaded in load_and_prepare_data
    # Get a sample customer_id
    if not df.empty:
        sample_customer_id = df['Customer_id'].iloc[0]
        # Select the row corresponding to the sample customer
        sample_customer_data = df[df['Customer_id'] == sample_customer_id]
        predictions = trainer.predict_single_customer(sample_customer_data)

        if predictions:
            print(f"Customer: {sample_customer_id}")
            for i, pred in enumerate(predictions, 1):
                print(f"  {i}. {pred['channel']}: {pred['score']:.4f}")
    else:
        print("Cannot perform demo prediction as dataframe is empty.")


    print("\n" + "="*70)
    print(" LIGHTGBM TRAINING COMPLETED!")
    print(f" Final NDCG Score: {ndcg:.4f}")
    print(" Model saved: lightgbm_channel_ranker.pkl")
    print("="*70)

if __name__ == "__main__":
    main()

 LIGHTGBM CHANNEL PREFERENCE RANKER TRAINING
 Loading data...
 Loaded 100,000 customers with 103 features
🔧 Encoding 9 categorical columns...
Converting to ranking format...
   Processed 1,000 customers...
   Processed 2,000 customers...
   Processed 3,000 customers...
   Processed 4,000 customers...
   Processed 5,000 customers...
   Processed 6,000 customers...
   Processed 7,000 customers...
   Processed 8,000 customers...
   Processed 9,000 customers...
   Processed 10,000 customers...
   Processed 11,000 customers...
   Processed 12,000 customers...
   Processed 13,000 customers...
   Processed 14,000 customers...
   Processed 15,000 customers...
   Processed 16,000 customers...
   Processed 17,000 customers...
   Processed 18,000 customers...
   Processed 19,000 customers...
   Processed 20,000 customers...
   Processed 21,000 customers...
   Processed 22,000 customers...
   Processed 23,000 customers...
   Processed 24,000 customers...
   Processed 25,000 customers...
   Process

In [14]:
import pandas as pd
import numpy as np
import shap
import joblib
import warnings

warnings.filterwarnings("ignore")


def compute_feature_shap_for_dataset_all_features(
    model_path="lightgbm_channel_ranker.pkl",
    data_path="features_with_labels.csv",
    max_customers=15000,
    save_path="mean_shap_feature_scores.csv"
):
    """
    Computes mean SHAP scores for ALL features in features_with_labels.csv,
    including features that were not used in the model (they will get score 0).
    """

    print("\nLoading model artifacts...")
    artifacts = joblib.load(model_path)
    model = artifacts["model"]
    label_encoders = artifacts["label_encoders"]
    channels = artifacts["channels"]

    print("Loading dataset...")
    df = pd.read_csv(data_path)

    # --- Identify ALL base features from CSV ---
    exclude_cols = ['Customer_id', 'Channel_Preference_Order', 'Preference_Label', 'Top_Channel']
    exclude_cols.extend([col for col in df.columns if 'Prefers_' in col])

    all_base_features = [col for col in df.columns if col not in exclude_cols]
    print(f"Found {len(all_base_features)} total base features in CSV.")

    # --- Get features actually used by the model ---
    if hasattr(model, 'feature_name_'):
        model_features = model.feature_name_
    else:
        # For LightGBM, we can reconstruct feature names
        num_model_features = model.num_feature()
        # We know the model has base features + channel one-hots
        model_base_feature_count = num_model_features - len(channels)
        # We don't know exact names, so we'll use what we have
        model_features = all_base_features[:model_base_feature_count] if len(all_base_features) >= model_base_feature_count else all_base_features

    print(f"Model uses {len(model_features)} base features.")

    # --- Prepare data for SHAP computation ---
    base_df = df[all_base_features].copy()
    n_customers = base_df.shape[0]

    # Sample if needed
    if n_customers > max_customers:
        idx = np.random.choice(n_customers, size=max_customers, replace=False)
        base_df = base_df.iloc[idx].reset_index(drop=True)
        print(f"Sampled down to {max_customers} customers for SHAP computation.")
    else:
        print(f"Using all {n_customers} customers for SHAP computation.")

    # Encode categorical features
    print("\nEncoding categorical columns...")
    for col in base_df.select_dtypes(include=["object"]).columns:
        if col in label_encoders:
            le = label_encoders[col]
            try:
                base_df[col] = le.transform(base_df[col].astype(str))
            except Exception as e:
                print(f"  Warning: unseen categories in '{col}', filling with -1")
                base_df[col] = -1
        else:
            base_df[col] = -1

    # Convert to numeric, filling non-numeric with 0
    for col in base_df.columns:
        base_df[col] = pd.to_numeric(base_df[col], errors='coerce').fillna(0)

    base_features_array = base_df.values
    n_customers_sample = base_features_array.shape[0]

    # --- Build ranking-style matrix ---
    print("\nReconstructing ranking-style feature matrix...")
    ranking_rows = []

    for i in range(n_customers_sample):
        customer_features = base_features_array[i]
        for ch in channels:
            ch_one_hot = np.zeros(len(channels))
            ch_idx = channels.index(ch)
            ch_one_hot[ch_idx] = 1
            combined = np.concatenate([customer_features, ch_one_hot])
            ranking_rows.append(combined)

    X_ranking = np.array(ranking_rows)
    print(f"Ranking matrix shape: {X_ranking.shape}")

    # --- Create feature name mapping ---
    all_feature_names = all_base_features + [f"Channel_{ch}_onehot" for ch in channels]

    # --- Compute SHAP values ---
    print("\nComputing SHAP values...")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_ranking)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    print(f"SHAP values shape: {shap_values.shape}")

    # --- Create results for ALL base features ---
    results = []

    # For features used by the model
    used_features_set = set(model_features)
    shap_base = shap_values[:, :len(model_features)]  # Only the portion used by model

    for i, feature in enumerate(model_features):
        if i < shap_base.shape[1]:  # Safety check
            mean_abs_shap = np.abs(shap_base[:, i]).mean()
            results.append({"feature": feature, "mean_abs_shap": mean_abs_shap, "used_in_model": True})

    # For features NOT used by model (get score 0)
    for feature in all_base_features:
        if feature not in used_features_set:
            results.append({"feature": feature, "mean_abs_shap": 0.0, "used_in_model": False})

    # Create final DataFrame
    shap_df = pd.DataFrame(results).sort_values("mean_abs_shap", ascending=False)

    print(f"\nSHAP scores computed for {len(shap_df)} features:")
    print(f"- {len(shap_df[shap_df['used_in_model']])} features used in model")
    print(f"- {len(shap_df[~shap_df['used_in_model']])} features not used in model (score=0)")

    print("\nTop 20 features by mean |SHAP|:")
    print(shap_df.head(20))

    shap_df.to_csv(save_path, index=False)
    print(f"\n✅ Saved mean SHAP scores for ALL features to: {save_path}")

    return shap_df


# ---- RUN ----
if __name__ == "__main__":
    shap_scores = compute_feature_shap_for_dataset_all_features()


Loading model artifacts...
Loading dataset...
Found 99 total base features in CSV.
Model uses 99 base features.
Sampled down to 15000 customers for SHAP computation.

Encoding categorical columns...

Reconstructing ranking-style feature matrix...
Ranking matrix shape: (90000, 105)

Computing SHAP values...
SHAP values shape: (90000, 105)

SHAP scores computed for 99 features:
- 99 features used in model
- 0 features not used in model (score=0)

Top 20 features by mean |SHAP|:
                          feature  mean_abs_shap  used_in_model
95             Call_Effectiveness       0.434997           True
94            Email_Effectiveness       0.331323           True
93              SMS_Effectiveness       0.319982           True
96         WhatsApp_Effectiveness       0.299623           True
97              IVR_Effectiveness       0.287126           True
98      Field_Agent_Effectiveness       0.213836           True
13       channel_preference_score       0.105647           True
39    

In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import ndcg_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

class RandomForestRankerTrainer:
    """Train Random Forest model for channel preference ranking"""

    def __init__(self):
        self.model = None
        self.channels = ['SMS', 'Email', 'Call', 'WhatsApp', 'IVR', 'Field_Agent']
        self.label_encoders = {}

    def load_and_prepare_data(self, data_path):
        """Load and prepare data for Random Forest ranking"""
        print(" Loading data...")
        df = pd.read_csv(data_path)
        print(f" Loaded {len(df):,} customers with {len(df.columns):,} features")

        # Get feature columns (exclude label columns)
        exclude_cols = ['Customer_id', 'Channel_Preference_Order', 'Preference_Label', 'Top_Channel']
        exclude_cols.extend([col for col in df.columns if 'Prefers_' in col])
        feature_cols = [col for col in df.columns if col not in exclude_cols]

        # Encode categorical features
        X = df[feature_cols].copy()
        categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

        print(f"🔧 Encoding {len(categorical_cols)} categorical columns...")
        for col in categorical_cols:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            self.label_encoders[col] = le

        # Create ranking dataset
        print(" Converting to ranking format...")
        ranking_data = []
        group_sizes = []

        for idx, (_, row) in enumerate(df.iterrows()):
            if (idx + 1) % 1000 == 0:
                print(f"   Processed {idx + 1:,} customers...")

            customer_id = row['Customer_id']
            preference_order = row['Channel_Preference_Order'].split(',')

            customer_features = X.iloc[idx].values

            for rank, channel in enumerate(preference_order):
                if channel in self.channels:
                    # Channel one-hot encoding
                    channel_features = np.zeros(len(self.channels))
                    channel_idx = self.channels.index(channel)
                    channel_features[channel_idx] = 1

                    combined_features = np.concatenate([customer_features, channel_features])
                    relevance = len(self.channels) - rank  # Higher rank = higher relevance

                    ranking_data.append({
                        'customer_id': customer_id,
                        'channel': channel,
                        'features': combined_features,
                        'relevance': relevance,
                        'group_id': idx
                    })

            group_sizes.append(len(self.channels))

        X_ranking = np.array([item['features'] for item in ranking_data])
        y_ranking = np.array([item['relevance'] for item in ranking_data])
        groups = np.array(group_sizes)

        print(f" Ranking dataset: {len(X_ranking):,} samples, {len(groups):,} customer groups")
        return X_ranking, y_ranking, groups, df

    def train_model(self, X, y, groups, test_size=0.2, random_state=42):
        """Train Random Forest model"""
        print("\n Training Random Forest Ranker...")

        # Split data by groups (customers)
        n_train_groups = int(len(groups) * (1 - test_size))
        train_samples = sum(groups[:n_train_groups])

        X_train, X_test = X[:train_samples], X[train_samples:]
        y_train, y_test = y[:train_samples], y[train_samples:]
        groups_train, groups_test = groups[:n_train_groups], groups[n_train_groups:]

        print(f" Train set: {len(X_train):,} samples from {len(groups_train):,} customers")
        print(f" Test set:  {len(X_test):,} samples from {len(groups_test):,} customers")

        # Random Forest parameters
        rf_params = {
            'n_estimators': 100,
            'max_depth': 10,
            'min_samples_split': 5,
            'min_samples_leaf': 2,
            'max_features': 'sqrt',
            'bootstrap': True,
            'random_state': random_state,
            'n_jobs': -1  # Use all available cores
        }

        # Train Random Forest
        print("⏳ Training Random Forest...")
        self.model = RandomForestRegressor(**rf_params)
        self.model.fit(X_train, y_train)

        print(" Random Forest training completed!")
        return X_test, y_test, groups_test

    def evaluate_model(self, X_test, y_test, groups_test):
        """Evaluate the trained model"""
        print("\n Evaluating model performance...")

        # Make predictions
        y_pred = self.model.predict(X_test)

        # Calculate NDCG
        ndcg = self.calculate_ndcg(y_test, y_pred, groups_test)
        print(f" NDCG Score: {ndcg:.4f}")

        # Calculate additional metrics
        mse = np.mean((y_test - y_pred) ** 2)
        rmse = np.sqrt(mse)
        print(f" RMSE: {rmse:.4f}")

        return ndcg, y_pred, rmse

    def calculate_ndcg(self, y_true, y_pred, groups):
        """Calculate NDCG score for ranking evaluation"""
        y_true_reshaped = []
        y_pred_reshaped = []

        start_idx = 0
        for group_size in groups:
            end_idx = start_idx + group_size
            y_true_reshaped.append(y_true[start_idx:end_idx])
            y_pred_reshaped.append(y_pred[start_idx:end_idx])
            start_idx = end_idx

        return ndcg_score(y_true_reshaped, y_pred_reshaped)

    def save_model(self, model_path='random_forest_channel_ranker.pkl'):
        """Save the trained model and preprocessing objects"""
        model_artifacts = {
            'model': self.model,
            'label_encoders': self.label_encoders,
            'channels': self.channels
        }

        joblib.dump(model_artifacts, model_path)
        print(f" Model saved as '{model_path}'")

    def predict_single_customer(self, customer_data):
        """Predict channel preferences for a single customer"""
        if self.model is None:
            print(" Model not trained yet!")
            return None

        # Prepare customer features
        exclude_cols = ['Customer_id', 'Channel_Preference_Order', 'Preference_Label', 'Top_Channel']
        exclude_cols.extend([col for col in customer_data.columns if 'Prefers_' in col])
        feature_cols = [col for col in customer_data.columns if col not in exclude_cols]

        X_customer = customer_data[feature_cols].iloc[0].copy()

        # Encode categorical features
        categorical_cols = X_customer.select_dtypes(include=['object']).index.tolist()
        for col in categorical_cols:
            if col in self.label_encoders:
                le = self.label_encoders[col]
                X_customer[col] = le.transform([X_customer[col].astype(str)])[0]

        customer_features = X_customer.values

        # Create samples for each channel
        ranking_data = []
        for channel in self.channels:
            channel_features = np.zeros(len(self.channels))
            channel_idx = self.channels.index(channel)
            channel_features[channel_idx] = 1

            combined_features = np.concatenate([customer_features, channel_features])
            ranking_data.append({
                'channel': channel,
                'features': combined_features
            })

        X_ranking = np.array([item['features'] for item in ranking_data])

        # Make predictions
        predictions = self.model.predict(X_ranking)

        # Combine results
        channel_predictions = []
        for i, data in enumerate(ranking_data):
            channel_predictions.append({
                'channel': data['channel'],
                'score': predictions[i]
            })

        # Sort by score
        channel_predictions.sort(key=lambda x: x['score'], reverse=True)
        return channel_predictions

    def cross_validate_model(self, X, y, groups, n_splits=5):
        """Perform cross-validation to assess model stability"""
        print("\n Performing Cross-Validation...")

        from sklearn.model_selection import KFold
        from sklearn.metrics import mean_squared_error

        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        ndcg_scores = []
        rmse_scores = []

        fold = 1
        # Split on the indices of the groups
        for train_group_indices, test_group_indices in kf.split(np.arange(len(np.unique(groups)))):
            # Get actual group IDs for this fold
            all_unique_groups = np.unique(groups)
            train_groups = all_unique_groups[train_group_indices]
            test_groups = all_unique_groups[test_group_indices]

            # Get sample indices for these groups
            train_mask = np.isin(groups, train_groups)
            test_mask = np.isin(groups, test_groups)

            X_train, X_test = X[train_mask], X[test_mask]
            y_train, y_test = y[train_mask], y[test_mask]
            # Pass the original groups array filtered by test_mask for NDCG calculation
            groups_test_fold = groups[test_mask]


            # Train model on this fold
            model = RandomForestRegressor(
                n_estimators=50,  # Smaller for faster CV
                max_depth=8,
                random_state=42,
                n_jobs=-1
            )
            model.fit(X_train, y_train)

            # Predict and evaluate
            y_pred = model.predict(X_test)
            ndcg = self.calculate_ndcg(y_test, y_pred, groups_test_fold) # Use filtered groups
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))

            ndcg_scores.append(ndcg)
            rmse_scores.append(rmse)

            print(f"   Fold {fold}: NDCG = {ndcg:.4f}, RMSE = {rmse:.4f}")
            fold += 1

        print(f"\n Cross-Validation Results:")
        print(f"   Average NDCG: {np.mean(ndcg_scores):.4f} (±{np.std(ndcg_scores):.4f})")
        print(f"   Average RMSE: {np.mean(rmse_scores):.4f} (±{np.std(rmse_scores):.4f})")

        return ndcg_scores, rmse_scores

def main():
    """Main training function"""
    print("="*70)
    print(" RANDOM FOREST CHANNEL PREFERENCE RANKER TRAINING")
    print("="*70)

    # Initialize trainer
    trainer = RandomForestRankerTrainer()

    # Load and prepare data
    data_path = 'features_with_labels.csv'  # Your engineered features file
    X, y, groups, df = trainer.load_and_prepare_data(data_path)

    # Train model
    X_test, y_test, groups_test = trainer.train_model(X, y, groups)

    # Evaluate model
    ndcg, predictions, rmse = trainer.evaluate_model(X_test, y_test, groups_test)

    # Save model
    trainer.save_model()

    # Model summary
    print("\n" + "="*70)
    print(" RANDOM FOREST TRAINING COMPLETED!")
    print("="*70)
    print(f" Final NDCG Score: {ndcg:.4f}")
    print(f" Final RMSE: {rmse:.4f}")
    print(f" Number of Trees: {trainer.model.n_estimators}")
    print(f" Number of Features: {trainer.model.n_features_in_}")
    print(" Model saved: random_forest_channel_ranker.pkl")

    # Comparison with baseline
    baseline_ndcg = 0.5  # Random ranking baseline
    improvement = ((ndcg - baseline_ndcg) / baseline_ndcg) * 100
    print(f" Improvement over random: {improvement:+.1f}%")
    print("="*70)

if __name__ == "__main__":
    main()

 RANDOM FOREST CHANNEL PREFERENCE RANKER TRAINING
 Loading data...
 Loaded 100,000 customers with 103 features
🔧 Encoding 9 categorical columns...
 Converting to ranking format...
   Processed 1,000 customers...
   Processed 2,000 customers...
   Processed 3,000 customers...
   Processed 4,000 customers...
   Processed 5,000 customers...
   Processed 6,000 customers...
   Processed 7,000 customers...
   Processed 8,000 customers...
   Processed 9,000 customers...
   Processed 10,000 customers...
   Processed 11,000 customers...
   Processed 12,000 customers...
   Processed 13,000 customers...
   Processed 14,000 customers...
   Processed 15,000 customers...
   Processed 16,000 customers...
   Processed 17,000 customers...
   Processed 18,000 customers...
   Processed 19,000 customers...
   Processed 20,000 customers...
   Processed 21,000 customers...
   Processed 22,000 customers...
   Processed 23,000 customers...
   Processed 24,000 customers...
   Processed 25,000 customers...
   P

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ndcg_score
import joblib
import warnings
warnings.filterwarnings('ignore')

class ModelComparisonVisualizer:
    """Comprehensive comparison of XGBoost, LightGBM, and Random Forest models"""

    def __init__(self):
        self.models = {}
        self.results = {}
        self.metrics = {}

    def load_models(self):
        """Load all trained models"""
        print("📁 Loading trained models...")

        model_files = {
            'XGBoost': 'xgb_channel_ranker.pkl',
            'LightGBM': 'lightgbm_channel_ranker.pkl',
            'Random Forest': 'random_forest_channel_ranker.pkl'
        }

        for model_name, file_path in model_files.items():
            try:
                if model_name == 'XGBoost':
                    self.models[model_name] = joblib.load(file_path)
                else:
                    model_data = joblib.load(file_path)
                    self.models[model_name] = model_data['model']
                print(f"✅ {model_name} loaded successfully")
            except FileNotFoundError:
                print(f"❌ {model_name} model file not found: {file_path}")

        return len(self.models) > 0

    def load_test_data(self, data_path='features_with_labels.csv'):
        """Load and prepare test data for evaluation"""
        print("\n📊 Loading test data...")
        self.df = pd.read_csv(data_path)

        exclude_cols = ['Customer_id', 'Channel_Preference_Order', 'Preference_Label', 'Top_Channel']
        exclude_cols.extend([col for col in self.df.columns if 'Prefers_' in col])
        self.feature_cols = [col for col in self.df.columns if col not in exclude_cols]

        X, y, groups = self.prepare_ranking_data(self.df)

        n_test_groups = int(len(groups) * 0.2)
        test_samples = sum(groups[-n_test_groups:])

        self.X_test = X[-test_samples:]
        self.y_test = y[-test_samples:]
        self.groups_test = groups[-n_test_groups:]

        print(f"📈 Test set: {len(self.X_test):,} samples from {len(self.groups_test):,} customers")
        return True

    def prepare_ranking_data(self, df):
        """Prepare data in ranking format (same as training)"""
        from sklearn.preprocessing import LabelEncoder

        X = df[self.feature_cols].copy()
        categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
        label_encoders = {}

        for col in categorical_cols:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            label_encoders[col] = le

        channels = ['SMS', 'Email', 'Call', 'WhatsApp', 'IVR', 'Field_Agent']
        ranking_data = []
        group_sizes = []

        for idx, (_, row) in enumerate(df.iterrows()):
            preference_order = row['Channel_Preference_Order'].split(',')
            customer_features = X.iloc[idx].values

            for rank, channel in enumerate(preference_order):
                if channel in channels:
                    channel_features = np.zeros(len(channels))
                    channel_idx = channels.index(channel)
                    channel_features[channel_idx] = 1

                    combined_features = np.concatenate([customer_features, channel_features])
                    relevance = len(channels) - rank

                    ranking_data.append({
                        'features': combined_features,
                        'relevance': relevance,
                        'group_id': idx
                    })

            group_sizes.append(len(channels))

        X_ranking = np.array([item['features'] for item in ranking_data])
        y_ranking = np.array([item['relevance'] for item in ranking_data])
        groups = np.array(group_sizes)

        return X_ranking, y_ranking, groups

    def evaluate_models(self):
        """Evaluate all models on test data"""
        print("\n📈 Evaluating models on test data...")

        for model_name, model in self.models.items():
            print(f"\n🔍 Evaluating {model_name}...")

            y_pred = model.predict(self.X_test)

            # Only NDCG metric retained
            ndcg = self.calculate_ndcg(self.y_test, y_pred, self.groups_test)

            self.results[model_name] = {
                'predictions': y_pred,
                'ndcg': ndcg
            }

            print(f"   ✅ NDCG: {ndcg:.4f}")

        return self.results

    def calculate_ndcg(self, y_true, y_pred, groups):
        """Calculate NDCG score for ranking evaluation"""
        y_true_reshaped = []
        y_pred_reshaped = []

        start_idx = 0
        for group_size in groups:
            end_idx = start_idx + group_size
            y_true_reshaped.append(y_true[start_idx:end_idx])
            y_pred_reshaped.append(y_pred[start_idx:end_idx])
            start_idx = end_idx

        return ndcg_score(y_true_reshaped, y_pred_reshaped)

    def generate_comparison_report(self):
        """Generate a comprehensive comparison report"""
        print("\n" + "="*80)
        print("📊 COMPREHENSIVE MODEL COMPARISON REPORT")
        print("="*80)

        print("\n🏆 PERFORMANCE SUMMARY:")
        print("-" * 50)

        comparison_data = []
        for model_name, results in self.results.items():
            comparison_data.append({
                'Model': model_name,
                'NDCG': results['ndcg']
            })

        df_comparison = pd.DataFrame(comparison_data)
        df_comparison['Rank'] = df_comparison['NDCG'].rank(ascending=False)
        df_comparison = df_comparison.sort_values('Rank')

        print(df_comparison.to_string(index=False))

        best_model = df_comparison.iloc[0]
        print(f"\n🎯 RECOMMENDED MODEL: {best_model['Model']}")
        print(f"   • NDCG Score: {best_model['NDCG']:.4f}")
        print(f"   • Ranking: #{int(best_model['Rank'])}")

        df_comparison.to_csv('detailed_model_comparison_report.csv', index=False)
        print("\n💾 Detailed report saved to 'detailed_model_comparison_report.csv'")

def main():
    """Main comparison function"""

    visualizer = ModelComparisonVisualizer()

    if not visualizer.load_models():
        print("❌ Failed to load models. Please ensure all model files exist.")
        return

    if not visualizer.load_test_data():
        print("❌ Failed to load test data.")
        return

    visualizer.evaluate_models()
    visualizer.generate_comparison_report()

    print("\n" + "="*80)
    print("✅ MODEL COMPARISON COMPLETED!")
    print("="*80)

if __name__ == "__main__":
    main()


📁 Loading trained models...
✅ XGBoost loaded successfully
✅ LightGBM loaded successfully
✅ Random Forest loaded successfully

📊 Loading test data...
📈 Test set: 120,000 samples from 20,000 customers

📈 Evaluating models on test data...

🔍 Evaluating XGBoost...
   ✅ NDCG: 0.9991

🔍 Evaluating LightGBM...
   ✅ NDCG: 0.9998

🔍 Evaluating Random Forest...
   ✅ NDCG: 0.9761

📊 COMPREHENSIVE MODEL COMPARISON REPORT

🏆 PERFORMANCE SUMMARY:
--------------------------------------------------
        Model     NDCG  Rank
     LightGBM 0.999809   1.0
      XGBoost 0.999140   2.0
Random Forest 0.976060   3.0

🎯 RECOMMENDED MODEL: LightGBM
   • NDCG Score: 0.9998
   • Ranking: #1

💾 Detailed report saved to 'detailed_model_comparison_report.csv'

✅ MODEL COMPARISON COMPLETED!
